<a href="https://colab.research.google.com/github/MWANIKID/When-Does-Regime-Information-Improve-Volatility-Forecasting-/blob/main/When_Does_Regime_Information_Improve_Volatility_Forecasting%3F.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
NSE VOLATILITY FORECASTING — REVIEWER-ALIGNED PYTHON PIPELINE v5.09
==================================================================
Study period: 2016-08-01 to 2026-07-31
Expected sample: 52 firms, 11 sectors.

This script starts from the FINAL reviewer-aligned R outputs. It does NOT
reconstruct the authoritative return/variance target or primary econometric
forecasts. It adds the Python-stage recommendations: common chronological
splits; leakage-safe preprocessing; GRU, Transformer and PatchTST; five-seed
stability; hyperparameter tuning; regime classification; regime-calibrated
TGARCH-X; regime-aware Transformer; constrained-convex forecast combination;
sector/state/weighting evaluation; Parkinson robustness; dependence-corrected
inference; MCS/SPA; DM-HLN-Holm; GW/GR; Mincer-Zarnowitz; two-way clustered
moderation inference; wild-cluster bootstrap; 3-state regime robustness;
sequence and split sensitivity; high-volatility diagnostics; economic
significance; reproducibility and reviewer-coverage outputs.

Terminology:
  * "Regime-Calibrated TGARCH-X" = multiplicatively calibrated baseline TGARCH-X.
  * "Combined Forecast" = constrained-convex TGARCH-X/Transformer combination.
  * GRU and PatchTST are external benchmarks, not additional regime focal models.

Use QUICK_TEST=False for journal results. v5.09 runs the full journal analysis from scratch in one local session. It uses the same leakage-safe zero-neutral Student-t HMM and probability-aware forecasting architecture as v5.08, but disables Google Drive persistence and reuse of earlier neural outputs. The final reproducibility bundle contains the exact inputs, outputs, environment record, hashes and run metadata.
"""

from __future__ import annotations
import os, sys, gc, json, math, random, hashlib, platform, warnings
import subprocess, zipfile, re, shutil, time
from pathlib import Path
from typing import Dict, List, Tuple, Optional
from itertools import combinations

warnings.filterwarnings("ignore")

AUTO_INSTALL = True
QUICK_TEST = False

def _install_if_missing(package: str, import_name: Optional[str] = None):
    import importlib.util
    name = import_name or package
    if importlib.util.find_spec(name) is None:
        if not AUTO_INSTALL:
            raise ImportError(f"Missing package: {package}")
        subprocess.check_call([sys.executable, "-m", "pip", "install", package, "--quiet"])

for pkg, imp in [
    ("numpy","numpy"),("pandas","pandas"),("scipy","scipy"),
    ("scikit-learn","sklearn"),("statsmodels","statsmodels"),
    ("matplotlib","matplotlib"),("openpyxl","openpyxl"),
    ("tqdm","tqdm"),("arch","arch"),("patsy","patsy"),("numba","numba")
]:
    _install_if_missing(pkg, imp)

try:
    import tensorflow as tf
except ImportError:
    if not AUTO_INSTALL:
        raise
    subprocess.check_call([sys.executable, "-m", "pip", "install", "tensorflow>=2.15", "--quiet"])
    import tensorflow as tf

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from scipy.optimize import minimize_scalar, minimize
from scipy.stats import chi2
from scipy.special import logsumexp, expit
from tqdm import tqdm
from sklearn.model_selection import TimeSeriesSplit
import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.stats.multitest import multipletests
from statsmodels.stats.sandwich_covariance import cov_cluster_2groups
from tensorflow import keras
from tensorflow.keras import layers
from numba import njit

try:
    from arch.bootstrap import MCS, SPA
    ARCH_BOOTSTRAP_AVAILABLE = True
except Exception:
    ARCH_BOOTSTRAP_AVAILABLE = False

# =============================================================================
# 1. CONFIGURATION
# =============================================================================
CODE_VERSION = "nse_reviewer_aligned_python_v5_09_single_session_reproducible_2016_2026"

R_INPUT = Path("TGARCH_X_Reviewer_Aligned_20160801_20260731_PRIMARY.zip")
MASTER_FILE = Path("Final Master File_All variables 01082016-31072026.csv")

# In Google Colab, ALWAYS open the upload dialog at the start of each run so
# the analysis uses the files selected for that run rather than stale session
# copies. Set to False only if you intentionally want automatic file discovery.
COLAB_ALWAYS_UPLOAD = True

EXPECTED_R_CODE_VERSION = "tgarch_x_reviewer_aligned_v4_04_2016_2026"
EXPECTED_FIRMS = 52
EXPECTED_SECTORS = 11
SAMPLE_START = pd.Timestamp("2016-08-01")
SAMPLE_END = pd.Timestamp("2026-07-31")

OUTPUT_DIR = Path("NSE_Reviewer_Aligned_Python_Output_v5_09_SingleSession")
PLOT_DIR = OUTPUT_DIR / "plots"

# Do not permit a manuscript run to silently append to a partial earlier run.
if OUTPUT_DIR.exists():
    existing_files = [
        p for p in OUTPUT_DIR.rglob("*")
        if p.is_file() and not p.name.endswith(".tmp")
    ]
    if existing_files:
        raise RuntimeError(
            "Fresh single-session run blocked because the v5.09 output folder "
            "already contains files. Restart the Colab runtime for a clean "
            "journal run, or manually archive/delete that folder first."
        )

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
PLOT_DIR.mkdir(parents=True, exist_ok=True)

# -------------------------------------------------------------------------
# SINGLE-SESSION REPRODUCIBILITY SETTINGS
# -------------------------------------------------------------------------
# This version deliberately does NOT use Google Drive and does NOT reuse any
# result from v5.05-v5.08. Every reported estimate is generated from the two
# freshly uploaded inputs during this one execution.
RESUME_COMPLETED_OUTPUTS = False
PERSIST_CHECKPOINTS_TO_DRIVE = False
REQUIRE_GPU_FOR_FULL_COLAB_RUN = True
ALLOW_CPU_FULL_RUN = False

# Local checkpoints are written only as crash diagnostics inside this runtime.
# They are never used to import results from an earlier run/session.
LOCAL_CHECKPOINTS = True
PERSISTENT_OUTPUT_DIR = None
CHECKPOINT_DIR = OUTPUT_DIR / "_checkpoints_v5_09"
RESUME_SEARCH_DIRS = [OUTPUT_DIR]
LEGACY_CHECKPOINT_DIRS = []

# A clean-run guard prevents accidental mixing of a previous partial v5.09
# execution with a new journal run.
FRESH_SINGLE_SESSION_RUN = True

# At completion, bundle exact inputs + outputs + environment metadata together.
INCLUDE_EXACT_INPUTS_IN_REPRO_BUNDLE = True
AUTO_DOWNLOAD_REPRO_BUNDLE_IN_COLAB = True

EPSILON = 1e-8
FORECAST_UPPER_CLIP = 1e3

# Global calendar splits. StackTrain remains genuinely out-of-sample for
# combination weights and calibration.
FINALTEST_START_RATIO = 0.65
STACKTRAIN_SHARE_OF_PRETEST = 0.20
VALIDATION_SHARE_OF_FITPOOL = 0.20

MIN_TRAIN_SEQUENCES = 80
MIN_VAL_SEQUENCES = 20
MIN_STACK_OBS = 20
MIN_MARKOV_OBS = 120

# Primary regime-estimation controls.
# The final manuscript specification is a two-state state-independent
# zero-hurdle Student-t HMM. Exact zero returns contribute to a common
# zero-return probability and therefore do not mechanically identify a
# volatility state. Non-zero return magnitudes identify the latent states.
HMM_MODEL_SIGNATURE = "zero_neutral_student_t_hmm2_softprob_v1"
HMM_MIN_STATE_SHARE = 0.03
HMM_MIN_STATE_SHARE_3 = 0.02
HMM_RARE_STATE_WARNING_SHARE = 0.05
HMM_MIN_VARIANCE_RATIO = 1.20
HMM_ZERO_ATOL = 1e-12
HMM_TRANSITION_EPS = 1e-6

# Leakage-safe fitting:
# - tuning HMM: fit through Train_End, filter validation forward;
# - final HMM: refit through Validation_End, filter StackTrain/FinalTest forward.
HMM_TWO_STATE_STARTS = 6
HMM_THREE_STATE_STARTS = 8
HMM_MAXITER_2STATE = 2200
HMM_MAXITER_3STATE = 3200

# Zero-return/state validity screen, evaluated on the fit sample only.
RUN_ZERO_RETURN_REGIME_PREFLIGHT = True
STOP_ON_ZERO_RETURN_REGIME_FAILURE = True
ZERO_LOW_DOMINANCE_THRESHOLD = 0.70
LOW_ZERO_CAPTURE_THRESHOLD = 0.85
ZERO_REGIME_PHI_THRESHOLD = 0.50
ZERO_REGIME_ODDS_RATIO_THRESHOLD = 5.0
ZERO_LOW_WARNING_THRESHOLD = 0.50
ZERO_REGIME_PHI_WARNING_THRESHOLD = 0.35

# Probability-aware downstream specification.
REGIME_PROBABILITY_FEATURE_NAME = "Regime_Probability_High"
REGIME_COMBINATION_A_BOUNDS = (-8.0, 8.0)
REGIME_COMBINATION_B_BOUNDS = (-12.0, 12.0)

# Robustness switches.
RUN_ZERO_NEUTRAL_GAUSSIAN_ROBUSTNESS = True
RUN_CONVENTIONAL_STUDENTT_DIAGNOSTIC = True

PRIMARY_SEQUENCE_LENGTH = 20
SEQUENCE_SENSITIVITY = [10, 20, 40]
NEURAL_SEEDS = [11, 23, 42, 71, 101]
TUNING_SEED = 42
BATCH_SIZE = 64
MAX_EPOCHS = 40
EARLY_STOPPING_PATIENCE = 6

BOOTSTRAP_REPS = 5000
MCS_SPA_REPS = 5000
WILD_CLUSTER_REPS = 999
BLOCK_LENGTH = 10
MCS_SIZE = 0.10
ALPHA = 0.05

RUN_SEQUENCE_SENSITIVITY = True
RUN_3STATE_REGIME_ROBUSTNESS = True
RUN_DATE_FE_SENSITIVITY = True
RUN_WILD_CLUSTER_BOOTSTRAP = True
RUN_ECONOMIC_SIGNIFICANCE = True

if QUICK_TEST:
    NEURAL_SEEDS = [42]
    MAX_EPOCHS = 3
    EARLY_STOPPING_PATIENCE = 1
    BOOTSTRAP_REPS = 100
    MCS_SPA_REPS = 100
    WILD_CLUSTER_REPS = 99
    RUN_SEQUENCE_SENSITIVITY = False
    HMM_TWO_STATE_STARTS = 2
    HMM_THREE_STATE_STARTS = 2
    RUN_ZERO_NEUTRAL_GAUSSIAN_ROBUSTNESS = False
    RUN_CONVENTIONAL_STUDENTT_DIAGNOSTIC = False
    RUN_3STATE_REGIME_ROBUSTNESS = False

TRANSFORMER_GRID = [
    {"d_model":32,"n_heads":4,"d_ff":64,"dropout":0.15,"lr":1e-3},
    {"d_model":48,"n_heads":4,"d_ff":96,"dropout":0.10,"lr":5e-4},
]
GRU_GRID = [
    {"units":32,"dense":32,"dropout":0.15,"lr":1e-3},
    {"units":48,"dense":32,"dropout":0.10,"lr":5e-4},
]
PATCHTST_GRID = [
    {"patch_len":5,"d_model":32,"n_heads":4,"d_ff":64,"dropout":0.15,"lr":1e-3},
    {"patch_len":5,"d_model":48,"n_heads":4,"d_ff":96,"dropout":0.10,"lr":5e-4},
]

UNCONSTRAINED_WEIGHT_BOUNDS = (-2.0, 3.0)
CALIBRATION_SHRINKAGE_K = 25.0
CALIBRATION_FACTOR_BOUNDS = (0.25, 4.0)

R_FORECAST_COLUMNS = {
    "EWMA":"EWMA_Forecast",
    "Historical Variance":"Historical_Variance_Forecast",
    "TGARCH-X":"TGARCH_X_Forecast",
    "TGARCH":"TGARCH_Forecast_NoX",
    "GJR-GARCH":"GJR_GARCH_Forecast",
    "GARCH":"GARCH_Forecast",
}
PRIMARY_MODEL_COLS = {
    "Baseline TGARCH-X":"FC_TGARCH_X",
    "Baseline Transformer":"FC_Transformer",
    "Static Combined Forecast":"FC_Combined_Static",
    "Regime-Calibrated TGARCH-X":"FC_TGARCH_X_RegCal",
    "Regime-Aware Transformer":"FC_Transformer_Regime",
    "Regime-Aware Combined Forecast":"FC_Combined_Regime",
}
WEIGHT_SCHEMES = {
    "MarketCapitalisation":"Market_Weight_Daily",
    "EqualSector":"Equal_Weight_Daily",
    "CappedMarketCapitalisation":"Capped_Market_Weight_Daily",
}

# =============================================================================
# 2. HELPERS
# =============================================================================
def set_seed(seed:int):
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed); np.random.seed(seed); tf.keras.utils.set_random_seed(seed)
    try: tf.config.experimental.enable_op_determinism()
    except Exception: pass

set_seed(TUNING_SEED)
try:
    for gpu in tf.config.list_physical_devices("GPU"):
        tf.config.experimental.set_memory_growth(gpu, True)
except Exception:
    pass
tf.keras.mixed_precision.set_global_policy("float32")

def _atomic_csv_write(df, path: Path, index=False):
    """Write via a temporary file so a disconnect cannot leave a half CSV."""
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + ".tmp")
    df.to_csv(tmp, index=index)
    tmp.replace(path)
    return path

def save_csv(df, name, index=False):
    """Save one authoritative local copy inside the single-session output tree."""
    return _atomic_csv_write(df, OUTPUT_DIR / name, index=index)

def _safe_slug(value):
    return re.sub(r"[^A-Za-z0-9._-]+", "_", str(value)).strip("_")

def _resume_file(name: str):
    """
    Return the newest existing copy of an output across local output,
    v5.05 persistent storage, and legacy Drive backup folders.
    """
    if not RESUME_COMPLETED_OUTPUTS:
        return None
    candidates = []
    for root in RESUME_SEARCH_DIRS:
        try:
            p = Path(root) / name
            if p.exists() and p.is_file() and p.stat().st_size > 0:
                candidates.append(p)
        except Exception:
            pass
    if not candidates:
        return None
    return max(candidates, key=lambda p: p.stat().st_mtime)

def _read_resume_csv(name: str, parse_date_cols=("Date",)):
    p = _resume_file(name)
    if p is None:
        return None
    try:
        df = pd.read_csv(p, low_memory=False)
        for c in parse_date_cols:
            if c in df.columns:
                df[c] = parse_dates(df[c])
        print(f"RESUME: loaded {name} from {p}")
        return df
    except Exception as e:
        print(f"WARNING: resume file {p} could not be read: {e}")
        return None

def _sync_existing_local_outputs_to_drive():
    """
    Preserve completed v5.04/v5.05 outputs before any new expensive training.
    Only non-temporary files are copied. Existing newer Drive copies win.
    """
    if PERSISTENT_OUTPUT_DIR is None or not OUTPUT_DIR.exists():
        return
    dest_root = Path(PERSISTENT_OUTPUT_DIR)
    dest_root.mkdir(parents=True, exist_ok=True)
    copied = 0
    for p in OUTPUT_DIR.rglob("*"):
        if not p.is_file() or p.name.endswith(".tmp"):
            continue
        rel = p.relative_to(OUTPUT_DIR)
        dest = dest_root / rel
        try:
            dest.parent.mkdir(parents=True, exist_ok=True)
            if (not dest.exists()) or p.stat().st_mtime > dest.stat().st_mtime:
                shutil.copy2(p, dest)
                copied += 1
        except Exception as e:
            print(f"WARNING: could not preserve {p}: {e}")
    print(f"Persistent backup sync complete: {copied} file(s) copied/updated.")

def _checkpoint_paths(stage: str, model_name: str, seq_len: int, sector: str):
    root = Path(CHECKPOINT_DIR) / _safe_slug(stage) / (
        f"{_safe_slug(model_name)}_seq{int(seq_len)}"
    )
    root.mkdir(parents=True, exist_ok=True)
    slug = _safe_slug(sector)
    return {
        "ensemble": root / f"{slug}_ensemble.csv",
        "tuning": root / f"{slug}_tuning.csv",
        "details": root / f"{slug}_details.csv",
        "done": root / f"{slug}.done.json",
    }

def _write_sector_checkpoint(
    stage: str,
    model_name: str,
    seq_len: int,
    sector: str,
    ensemble_df: pd.DataFrame,
    tuning_df: pd.DataFrame,
    details_df: pd.DataFrame,
):
    paths = _checkpoint_paths(stage, model_name, seq_len, sector)
    _atomic_csv_write(ensemble_df, paths["ensemble"], index=False)
    _atomic_csv_write(tuning_df, paths["tuning"], index=False)
    _atomic_csv_write(details_df, paths["details"], index=False)
    meta = {
        "code_version": CODE_VERSION,
        "stage": stage,
        "model": model_name,
        "sequence_length": int(seq_len),
        "sector": sector,
        "expected_seeds": list(map(int, NEURAL_SEEDS)),
        "ensemble_rows": int(len(ensemble_df)),
        "tuning_rows": int(len(tuning_df)),
        "details_rows": int(len(details_df)),
        "completed_at_unix": time.time(),
    }
    tmp = paths["done"].with_suffix(".json.tmp")
    with open(tmp, "w", encoding="utf-8") as f:
        json.dump(meta, f, indent=2)
    tmp.replace(paths["done"])

def _load_sector_checkpoint(stage: str, model_name: str, seq_len: int, sector: str):
    """
    v5.09 is a from-scratch single-session manuscript run.
    No current or legacy checkpoint is ever loaded into estimation.
    """
    return None


def sha256_file(path:Path):
    h=hashlib.sha256()
    with open(path,"rb") as f:
        for chunk in iter(lambda:f.read(1024*1024),b""): h.update(chunk)
    return h.hexdigest()

def parse_dates(s):
    """
    Parse both authoritative R ISO dates (YYYY-MM-DD) and master-file dates
    (DD/MM/YYYY) without corrupting valid ISO dates.
    """
    raw = s.astype("string").str.strip()

    # R outputs: ISO YYYY-MM-DD.
    out = pd.to_datetime(raw, format="%Y-%m-%d", errors="coerce")

    # Master file: DD/MM/YYYY.
    missing = out.isna() & raw.notna() & raw.ne("")
    if missing.any():
        out.loc[missing] = pd.to_datetime(
            raw.loc[missing],
            format="%d/%m/%Y",
            errors="coerce"
        )

    # Last-resort parser for any remaining non-empty strings.
    missing = out.isna() & raw.notna() & raw.ne("")
    if missing.any():
        out.loc[missing] = pd.to_datetime(
            raw.loc[missing],
            errors="coerce",
            dayfirst=True
        )

    return out

def clean_sector_names(s):
    return (s.astype(str).str.strip().replace({
        "Commercial and services":"Commercial and Services",
        "Commercial & Services":"Commercial and Services",
        "Telecommunication & Technology":"Telecommunication and Technology",
    }))

def qlike(y,f):
    y=np.clip(np.asarray(y,float),EPSILON,None)
    f=np.clip(np.asarray(f,float),EPSILON,None)
    r=y/f
    return r-np.log(r)-1.0

def weighted_mean_safe(values,weights):
    v=np.asarray(values,float); w=np.asarray(weights,float)
    m=np.isfinite(v)&np.isfinite(w)&(w>=0)
    if not m.any() or w[m].sum()<=0: return np.nan
    return float(np.average(v[m],weights=w[m]))

def moving_block_indices(n,block_length,rng):
    if n<=block_length: return rng.integers(0,n,size=n)
    starts=rng.integers(0,n-block_length+1,size=int(np.ceil(n/block_length)))
    return np.concatenate([np.arange(s,s+block_length) for s in starts])[:n]

def _candidate_roots():
    """Common notebook/Colab locations searched for required inputs."""
    roots = [
        Path.cwd(),
        Path("/content"),
        Path("/mnt/data"),
        Path("/content/drive/MyDrive"),
    ]
    out, seen = [], set()
    for root in roots:
        if not root.exists():
            continue
        try:
            key = str(root.resolve())
        except Exception:
            key = str(root)
        if key not in seen:
            out.append(root)
            seen.add(key)
    return out

def _find_input(path:Path, pattern:str):
    """
    Resolve an explicit path first. Otherwise search common Jupyter/Colab
    locations recursively so the notebook working directory does not have
    to equal the upload directory.
    """
    path = Path(path)
    if path.exists():
        return path

    cand = []
    for root in _candidate_roots():
        try:
            cand.extend(root.glob(pattern))
            cand.extend(root.rglob(pattern))
        except Exception:
            pass

    # De-duplicate and retain only existing candidates.
    dedup, seen = [], set()
    for c in cand:
        if not c.exists():
            continue
        try:
            key = str(c.resolve())
        except Exception:
            key = str(c)
        if key not in seen:
            dedup.append(c)
            seen.add(key)

    if len(dedup) == 1:
        print(f"Auto-detected input: {dedup[0]}")
        return dedup[0]

    if len(dedup) > 1:
        exact = [c for c in dedup if c.name == path.name]
        pool = exact if exact else dedup
        chosen = max(pool, key=lambda p: p.stat().st_mtime)
        print("Multiple input candidates found:")
        for c in dedup:
            print("  ", c)
        print(f"Using: {chosen}")
        return chosen

    searched = "\n".join(f"  - {r}" for r in _candidate_roots())
    raise FileNotFoundError(
        f"Could not locate required input '{path.name}'.\n"
        f"Searched:\n{searched}\n\n"
        "If using Google Colab, upload the ZIP and master CSV to /content "
        "with the Files panel or files.upload(), or set R_INPUT and "
        "MASTER_FILE to their exact paths."
    )

def _running_in_colab():
    try:
        import google.colab  # noqa: F401
        return True
    except Exception:
        return False

def _resolve_required_inputs():
    """
    In Google Colab, always prompt for fresh uploads when
    COLAB_ALWAYS_UPLOAD=True. Outside Colab, use the normal input-discovery
    search. This prevents stale files from an earlier notebook run being used
    silently.
    """
    global R_INPUT, MASTER_FILE

    if _running_in_colab() and COLAB_ALWAYS_UPLOAD:
        print("\nA Google Colab upload window will open now.")
        print("Please select BOTH current analysis files:")
        print("  1. TGARCH_X_Reviewer_Aligned_20160801_20260731_PRIMARY.zip")
        print("  2. Final Master File_All variables 01082016-31072026.csv\n")

        from google.colab import files
        uploaded = files.upload()

        if not uploaded:
            raise FileNotFoundError(
                "No files were uploaded. Upload both required input files."
            )

        print("\nUploaded files for this run:")
        uploaded_names = list(uploaded.keys())
        for fname in uploaded_names:
            print("  ", fname)

        # IMPORTANT: use the files selected in THIS upload dialog directly.
        # Colab may rename a newly uploaded duplicate to "(1)", "(2)", etc.
        # Calling the general discovery routine here could otherwise select an
        # older unsuffixed session copy.
        uploaded_paths = [Path(fname) for fname in uploaded_names]
        zip_candidates = [
            p for p in uploaded_paths
            if p.suffix.lower() == ".zip"
            and p.name.startswith(
                "TGARCH_X_Reviewer_Aligned_20160801_20260731_PRIMARY"
            )
        ]
        csv_candidates = [
            p for p in uploaded_paths
            if p.suffix.lower() == ".csv"
            and p.name.startswith("Final Master File_All variables")
        ]

        if len(zip_candidates) != 1 or len(csv_candidates) != 1:
            raise FileNotFoundError(
                "The upload must contain exactly one PRIMARY R ZIP and one "
                "Final Master CSV. Uploaded: " + ", ".join(uploaded_names)
            )

        R_INPUT = zip_candidates[0]
        MASTER_FILE = csv_candidates[0]

        if not R_INPUT.exists() or not MASTER_FILE.exists():
            raise FileNotFoundError(
                "Colab reported successful upload, but one selected file is "
                "not present in the current session."
            )

        print(f"Using freshly uploaded R ZIP: {R_INPUT}")
        print(f"Using freshly uploaded master CSV: {MASTER_FILE}")
        return

    # Non-Colab or explicit no-prompt mode.
    R_INPUT = _find_input(
        R_INPUT,
        "TGARCH_X_Reviewer_Aligned_20160801_20260731_PRIMARY*"
    )
    MASTER_FILE = _find_input(
        MASTER_FILE,
        "Final Master File_All variables*"
    )

_resolve_required_inputs()

# Preserve the exact analysis inputs inside the final reproducibility package.
INPUT_BUNDLE_DIR = OUTPUT_DIR / "inputs"
INPUT_BUNDLE_DIR.mkdir(parents=True, exist_ok=True)

if INCLUDE_EXACT_INPUTS_IN_REPRO_BUNDLE:
    bundled_r_input = INPUT_BUNDLE_DIR / R_INPUT.name
    bundled_master = INPUT_BUNDLE_DIR / MASTER_FILE.name
    shutil.copy2(R_INPUT, bundled_r_input)
    shutil.copy2(MASTER_FILE, bundled_master)
    print("Exact inputs copied into reproducibility bundle:")
    print("  ", bundled_r_input)
    print("  ", bundled_master)

# Record run start and package/runtime environment before estimation.
ENV_DIR = OUTPUT_DIR / "environment"
ENV_DIR.mkdir(parents=True, exist_ok=True)

with open(ENV_DIR / "RUN_STARTED_UTC.txt", "w", encoding="utf-8") as f:
    f.write(pd.Timestamp.utcnow().isoformat())

try:
    freeze_txt = subprocess.check_output(
        [sys.executable, "-m", "pip", "freeze"],
        text=True,
        stderr=subprocess.STDOUT,
    )
except Exception as e:
    freeze_txt = "pip freeze unavailable: " + repr(e)

with open(ENV_DIR / "pip_freeze.txt", "w", encoding="utf-8") as f:
    f.write(freeze_txt)

with open(ENV_DIR / "runtime_summary.txt", "w", encoding="utf-8") as f:
    f.write(f"Python: {sys.version}\n")
    f.write(f"Platform: {platform.platform()}\n")
    f.write(f"TensorFlow: {tf.__version__}\n")
    f.write(f"Code version: {CODE_VERSION}\n")
    f.write(f"GPU devices: {tf.config.list_physical_devices('GPU')}\n")

# Preserve the exact script when run as a .py file (recommended via %run).
try:
    source_path = Path(__file__).resolve()
    if source_path.exists() and source_path.is_file():
        shutil.copy2(
            source_path,
            ENV_DIR / "NSE_Reviewer_Aligned_Python_v5_09_SingleSession_Reproducible.py"
        )
except Exception as e:
    with open(ENV_DIR / "SOURCE_CODE_COPY_NOTICE.txt", "w", encoding="utf-8") as f:
        f.write(
            "The run was not executed from a resolvable .py file, so an automatic "
            "source-code copy could not be made. Run this script with %run to "
            "bundle the exact source automatically.\n" + repr(e)
        )

def _initialise_persistent_resume_storage():
    """
    Local-only single-session setup. Google Drive is intentionally not mounted.
    """
    global PERSISTENT_OUTPUT_DIR, CHECKPOINT_DIR, RESUME_SEARCH_DIRS, LEGACY_CHECKPOINT_DIRS

    PERSISTENT_OUTPUT_DIR = None
    CHECKPOINT_DIR = OUTPUT_DIR / "_checkpoints_v5_09"
    CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
    RESUME_SEARCH_DIRS = [OUTPUT_DIR]
    LEGACY_CHECKPOINT_DIRS = []

    print("Single-session mode: Google Drive persistence is DISABLED.")
    print("Single-session mode: prior v5.05-v5.08 outputs will NOT be reused.")
    print("Local output directory:", OUTPUT_DIR.resolve())


def _verify_compute_device():
    gpus = tf.config.list_physical_devices("GPU")
    if gpus:
        print("GPU detected:", ", ".join(d.name for d in gpus))
        return
    print("WARNING: no GPU detected; TensorFlow will use CPU.")
    if (
        _running_in_colab()
        and REQUIRE_GPU_FOR_FULL_COLAB_RUN
        and not QUICK_TEST
        and not ALLOW_CPU_FULL_RUN
    ):
        raise RuntimeError(
            "Full single-session journal run blocked because no GPU is active. "
            "Choose Runtime > Change runtime type > T4 GPU (or better), restart "
            "the runtime, and rerun v5.09 from the beginning. No Google Drive "
            "or prior-session output is used in this version."
        )

_initialise_persistent_resume_storage()
_verify_compute_device()

def read_r_csv(filename):
    if R_INPUT.is_dir():
        m=list(R_INPUT.rglob(filename))
        if len(m)!=1: raise FileNotFoundError(f"{filename}: matches={m}")
        return pd.read_csv(m[0],low_memory=False)
    if R_INPUT.suffix.lower()==".zip":
        with zipfile.ZipFile(R_INPUT,"r") as z:
            m=[n for n in z.namelist() if n.endswith("/"+filename) or n==filename]
            if len(m)!=1: raise FileNotFoundError(f"{filename}: matches={m}")
            with z.open(m[0]) as f: return pd.read_csv(f,low_memory=False)
    raise ValueError("R_INPUT must be a ZIP or extracted directory.")

# =============================================================================
# 3. LOAD AND VERIFY FINAL R OUTPUTS
# =============================================================================
print("="*100); print("LOADING AUTHORITATIVE R OUTPUTS"); print("="*100)
target=read_r_csv("20_Common_Target_for_Python.csv")
econ=read_r_csv("21_Econometric_Primary_Forecasts_Wide_for_Python.csv")
tgclean=read_r_csv("TGARCH_X_Forecasts_CLEAN.csv")
r_fit=read_r_csv("12_Fit_Coverage_and_Quality.csv")
r_fail=read_r_csv("13_Failure_Reasons.csv")
r_imp=read_r_csv("13A_TGARCH_X_Imputation_Audit.csv")
r_sparse=read_r_csv("05B_Sector_Trading_Sparsity_Audit.csv")
r_config=read_r_csv("29_Run_Configuration.csv")

for df_name, df in [("target",target),("econ",econ),("tgclean",tgclean)]:
    df["Date"]=parse_dates(df["Date"])
    df["Sector"]=clean_sector_names(df["Sector"])
    n_bad_dates=int(df["Date"].isna().sum())
    if n_bad_dates>0:
        raise ValueError(
            f"{df_name}: {n_bad_dates} Date values could not be parsed. "
            "Inspect the raw input before continuing."
        )

cfg=dict(zip(r_config["Setting"].astype(str),r_config["Value"].astype(str)))
r_version=cfg.get("CODE_VERSION","")
if r_version!=EXPECTED_R_CODE_VERSION:
    raise ValueError(f"R version mismatch: expected {EXPECTED_R_CODE_VERSION}, got {r_version}")
target_dup=target.duplicated(["Sector","Date"],keep=False)
econ_dup=econ.duplicated(["Sector","Date"],keep=False)
if target_dup.any() or econ_dup.any():
    target_dup_keys=target.loc[target_dup,["Sector","Date"]].drop_duplicates()
    econ_dup_keys=econ.loc[econ_dup,["Sector","Date"]].drop_duplicates()
    raise ValueError(
        "Duplicate Sector-Date rows remain after date parsing. "
        f"Target duplicate keys={len(target_dup_keys)}, "
        f"econometric duplicate keys={len(econ_dup_keys)}. "
        "These are genuine duplicates and should be inspected rather than dropped."
    )
if target["Sector"].nunique()!=EXPECTED_SECTORS:
    raise ValueError(f"Expected {EXPECTED_SECTORS} sectors.")

for _,wcol in WEIGHT_SCHEMES.items():
    if wcol not in target: raise ValueError(f"Missing weight: {wcol}")
    target[wcol]=pd.to_numeric(target[wcol],errors="coerce")

# Re-normalise only for numerical precision / varying active sector set.
for wcol in WEIGHT_SCHEMES.values():
    sums=target.groupby("Date")[wcol].transform("sum")
    target.loc[sums>0,wcol]=target.loc[sums>0,wcol]/sums[sums>0]

chk=econ[["Date","Sector","TGARCH_X_Forecast"]].merge(
    tgclean[["Date","Sector","TGARCH_Forecast"]],on=["Date","Sector"],how="inner",validate="one_to_one")
maxdiff=float(np.nanmax(np.abs(chk["TGARCH_X_Forecast"]-chk["TGARCH_Forecast"])))
if maxdiff>1e-10: raise ValueError(f"TGARCH-X mismatch across R exports: {maxdiff}")

save_csv(pd.DataFrame([
    {"Item":"R_Code_Version","Value":r_version},
    {"Item":"Target_Rows","Value":len(target)},
    {"Item":"Econometric_Rows","Value":len(econ)},
    {"Item":"Sectors","Value":target["Sector"].nunique()},
    {"Item":"Start","Value":target["Date"].min()},
    {"Item":"End","Value":target["Date"].max()},
    {"Item":"TGARCH_X_MaxCrossFileDiff","Value":maxdiff},
]),"00_Input_Audit.csv")
save_csv(r_fit,"00A_R_Fit_Coverage.csv")
save_csv(r_fail,"00B_R_Failures.csv")
save_csv(r_imp,"00C_R_TGARCH_X_Imputation_Audit.csv")
save_csv(r_sparse,"00D_R_Trading_Sparsity_Audit.csv")

# =============================================================================
# 4. MASTER AUDIT AND LEAKAGE-SAFE FEATURES
# =============================================================================
master=pd.read_csv(MASTER_FILE,low_memory=False)
master=master.loc[:,[c for c in master.columns if c and not str(c).startswith("Unnamed:")]]
req=["Date","Close","Volume","Category","Stock","Market Capitalization (KES)"]
miss=[c for c in req if c not in master]
if miss: raise ValueError(f"Master missing columns: {miss}")
master["Date"]=parse_dates(master["Date"])
master["Sector"]=clean_sector_names(master["Category"])
master["Stock"]=master["Stock"].astype(str).str.strip()
for c in ["Close","Volume","Market Capitalization (KES)"]:
    master[c]=pd.to_numeric(master[c],errors="coerce")
master=master[master["Date"].between(SAMPLE_START,SAMPLE_END)].copy()
core=(master["Date"].notna()&master["Sector"].ne("")&master["Stock"].ne("")&
      master["Close"].gt(0)&master["Market Capitalization (KES)"].gt(0))
mv=master[core].copy()
if mv["Stock"].nunique()!=EXPECTED_FIRMS: raise ValueError(f"Expected 52 firms, got {mv['Stock'].nunique()}")
if mv["Sector"].nunique()!=EXPECTED_SECTORS: raise ValueError(f"Expected 11 sectors, got {mv['Sector'].nunique()}")
if mv.duplicated(["Sector","Stock","Date"]).any(): raise ValueError("Duplicate Sector-Stock-Date rows.")

save_csv(pd.DataFrame([
    {"Metric":"Raw_rows","Value":len(master)},{"Metric":"Valid_core_rows","Value":len(mv)},
    {"Metric":"Unique_firms","Value":mv["Stock"].nunique()},
    {"Metric":"Unique_sectors","Value":mv["Sector"].nunique()},
]),"01_Master_Audit.csv")

sector_volume=(mv.groupby(["Sector","Date"],as_index=False)
               .agg(Sector_Volume=("Volume",lambda x:x.sum(min_count=1)),
                    Master_N_Stocks=("Stock","nunique")))
features=target.merge(sector_volume,on=["Sector","Date"],how="left",validate="one_to_one")
features=features.sort_values(["Sector","Date"]).reset_index(drop=True)
features["Log_Volume"]=np.log1p(features["Sector_Volume"].clip(lower=0))
features["Volume_Return"]=features.groupby("Sector")["Sector_Volume"].transform(
    lambda x:np.log(x.replace(0,np.nan)/x.replace(0,np.nan).shift(1)))
for w in [5,10,20]:
    features[f"Vol_{w}"]=features.groupby("Sector")["Sector_Return"].transform(
        lambda x:x.rolling(w,min_periods=w).std())
for c in ["Sector_Return","Actual_Variance","Log_Volume","Volume_Return",
          "Vol_5","Vol_10","Vol_20","Parkinson_Variance","N_Stocks"]:
    features[f"{c}_Lag1"]=features.groupby("Sector")[c].shift(1)

NEURAL_FEATURES=["Sector_Return_Lag1","Actual_Variance_Lag1","Log_Volume_Lag1",
                 "Volume_Return_Lag1","Vol_5_Lag1","Vol_10_Lag1","Vol_20_Lag1"]
for c in NEURAL_FEATURES:
    features[f"{c}_Missing"]=features[c].isna().astype(float)
NEURAL_FEATURES_ALL=NEURAL_FEATURES+[f"{c}_Missing" for c in NEURAL_FEATURES]
save_csv(features[["Date","Sector","Sector_Return","Actual_Variance","Parkinson_Variance"]+
                  list(WEIGHT_SCHEMES.values())+NEURAL_FEATURES_ALL],
         "02_Feature_Panel_PreScaling.csv")

# =============================================================================
# 5. COMMON GLOBAL DATE SPLITS
# =============================================================================
support_dates=np.array(sorted(pd.to_datetime(econ["Date"].dropna().unique())))
n_dates=len(support_dates)
if n_dates<200: raise RuntimeError("Too few common forecast dates.")
fi=int(np.floor(n_dates*FINALTEST_START_RATIO)); fi=min(max(fi,100),n_dates-30)
pre_dates=support_dates[:fi]; final_dates=support_dates[fi:]
ssi=int(np.floor(len(pre_dates)*(1-STACKTRAIN_SHARE_OF_PRETEST)))
ssi=min(max(ssi,60),len(pre_dates)-20)
fitpool_dates=pre_dates[:ssi]; stack_dates=pre_dates[ssi:]
vsi=int(np.floor(len(fitpool_dates)*(1-VALIDATION_SHARE_OF_FITPOOL)))
vsi=min(max(vsi,40),len(fitpool_dates)-15)
train_dates=fitpool_dates[:vsi]; val_dates=fitpool_dates[vsi:]

BOUND={
    "Train_End":pd.Timestamp(train_dates[-1]),
    "Validation_Start":pd.Timestamp(val_dates[0]),
    "Validation_End":pd.Timestamp(val_dates[-1]),
    "StackTrain_Start":pd.Timestamp(stack_dates[0]),
    "StackTrain_End":pd.Timestamp(stack_dates[-1]),
    "FinalTest_Start":pd.Timestamp(final_dates[0]),
    "FinalTest_End":pd.Timestamp(final_dates[-1]),
}
def split_label(d):
    d=pd.Timestamp(d)
    if d<=BOUND["Train_End"]: return "Train"
    if d<=BOUND["Validation_End"]: return "Validation"
    if d<=BOUND["StackTrain_End"]: return "StackTrain"
    return "FinalTest"

split_audit=pd.DataFrame([
    ["Train",train_dates[0],train_dates[-1],len(train_dates)],
    ["Validation",val_dates[0],val_dates[-1],len(val_dates)],
    ["StackTrain",stack_dates[0],stack_dates[-1],len(stack_dates)],
    ["FinalTest",final_dates[0],final_dates[-1],len(final_dates)],
],columns=["Split","Start","End","Unique_Dates"])
save_csv(split_audit,"03_Common_Date_Splits.csv")
print(split_audit.to_string(index=False))
features["Evaluation_Split"]=features["Date"].map(split_label)
econ["Evaluation_Split"]=econ["Date"].map(split_label)

# =============================================================================
# 6. SEQUENCE PREPARATION
# =============================================================================
def prepare_sector_matrix(sd,feature_cols,target_col,scaler_end_date):
    sd=sd.sort_values("Date").copy().reset_index(drop=True)
    train=sd[sd["Date"]<=scaler_end_date]
    med=train[NEURAL_FEATURES].median(numeric_only=True).fillna(0.0)
    for c in NEURAL_FEATURES: sd[c]=sd[c].fillna(med[c])
    Xraw=sd[feature_cols].astype(float).values
    smask=(sd["Date"]<=scaler_end_date).values
    mu=np.nanmean(Xraw[smask],axis=0); sig=np.nanstd(Xraw[smask],axis=0)
    mu=np.where(np.isfinite(mu),mu,0.0)
    sig=np.where(np.isfinite(sig)&(sig>EPSILON),sig,1.0)
    return sd,(Xraw-mu)/sig,np.column_stack([mu,sig])

def make_sequences(sd,X,target_col,seq_len):
    Xs=[]; ys=[]; dates=[]; rows=[]
    y=sd[target_col].astype(float).values; ds=sd["Date"].values
    for i in range(seq_len,len(sd)):
        if not np.isfinite(y[i]): continue
        win=X[i-seq_len:i]
        if not np.all(np.isfinite(win)): continue
        Xs.append(win); ys.append(max(float(y[i]),EPSILON))
        dates.append(pd.Timestamp(ds[i])); rows.append(i)
    return (np.asarray(Xs,np.float32),np.asarray(ys,np.float32),
            np.asarray(dates),np.asarray(rows,int))

# =============================================================================
# 7. GRU, TRANSFORMER AND CHANNEL-INDEPENDENT PATCHTST
# =============================================================================
@tf.function
def qlike_loss_tf(y_true,y_pred):
    yt=tf.clip_by_value(tf.cast(y_true,tf.float32),EPSILON,1e6)
    yp=tf.clip_by_value(tf.cast(y_pred,tf.float32),EPSILON,1e6)
    r=yt/yp
    return tf.reduce_mean(r-tf.math.log(r)-1.0)

def posenc(length,dmodel):
    pos=np.arange(length)[:,None]; i=np.arange(dmodel)[None,:]
    angle=pos/np.power(10000,(2*(i//2))/dmodel)
    pe=np.zeros((length,dmodel),np.float32)
    pe[:,0::2]=np.sin(angle[:,0::2]); pe[:,1::2]=np.cos(angle[:,1::2])
    return tf.constant(pe[None,:,:],tf.float32)

class TBlock(layers.Layer):
    def __init__(self,dmodel,nheads,dff,dropout=.1,attention_axes=None,**kw):
        super().__init__(**kw)
        self.mha=layers.MultiHeadAttention(num_heads=nheads,key_dim=max(1,dmodel//nheads),
                                           dropout=dropout,attention_axes=attention_axes)
        self.ff=keras.Sequential([layers.Dense(dff,activation="gelu"),
                                  layers.Dropout(dropout),layers.Dense(dmodel)])
        self.n1=layers.LayerNormalization(epsilon=1e-6)
        self.n2=layers.LayerNormalization(epsilon=1e-6)
        self.d1=layers.Dropout(dropout); self.d2=layers.Dropout(dropout)
    def call(self,x,training=False):
        a=self.mha(x,x,training=training)
        x=self.n1(x+self.d1(a,training=training))
        f=self.ff(x,training=training)
        return self.n2(x+self.d2(f,training=training))

def compile_model(m,lr):
    m.compile(optimizer=keras.optimizers.Adam(float(lr)),loss=qlike_loss_tf)
    return m

def build_gru(shape,hp):
    inp=keras.Input(shape=shape)
    x=layers.GRU(hp["units"],dropout=hp["dropout"],return_sequences=False)(inp)
    x=layers.Dense(hp["dense"],activation="gelu")(x); x=layers.Dropout(hp["dropout"])(x)
    out=layers.Dense(1,activation="softplus",dtype="float32")(x)
    return compile_model(keras.Model(inp,out),hp["lr"])

def build_transformer(shape,hp):
    sl,nf=shape; inp=keras.Input(shape=shape)
    x=layers.Dense(hp["d_model"])(inp); x=x+posenc(sl,hp["d_model"])
    x=TBlock(hp["d_model"],hp["n_heads"],hp["d_ff"],hp["dropout"])(x)
    x=layers.GlobalAveragePooling1D()(x)
    x=layers.Dense(hp["d_model"],activation="gelu")(x); x=layers.Dropout(hp["dropout"])(x)
    out=layers.Dense(1,activation="softplus",dtype="float32")(x)
    return compile_model(keras.Model(inp,out),hp["lr"])

def build_patchtst(shape,hp):
    sl,nf=shape; pl=int(hp["patch_len"])
    if sl%pl!=0: raise ValueError(f"seq_len={sl} must be divisible by patch_len={pl}")
    npatch=sl//pl
    inp=keras.Input(shape=shape)
    x=layers.Permute((2,1))(inp)                     # [B,C,L]
    x=layers.Reshape((nf,npatch,pl))(x)             # [B,C,P,patch]
    x=layers.Dense(hp["d_model"])(x)
    pe=posenc(npatch,hp["d_model"])
    pe=tf.reshape(pe,(1,1,npatch,hp["d_model"]))
    x=x+pe
    x=TBlock(hp["d_model"],hp["n_heads"],hp["d_ff"],hp["dropout"],
             attention_axes=(2,))(x)                # attention across patches
    x=layers.Lambda(lambda z:tf.reduce_mean(z,axis=2))(x)
    x=layers.Lambda(lambda z:tf.reduce_mean(z,axis=1))(x)
    x=layers.Dense(hp["d_model"],activation="gelu")(x); x=layers.Dropout(hp["dropout"])(x)
    out=layers.Dense(1,activation="softplus",dtype="float32")(x)
    return compile_model(keras.Model(inp,out),hp["lr"])

BUILDERS={"GRU":build_gru,"Transformer":build_transformer,"PatchTST":build_patchtst}
HPGRIDS={"GRU":GRU_GRID,"Transformer":TRANSFORMER_GRID,"PatchTST":PATCHTST_GRID}

# =============================================================================
# 8. CHRONOLOGICAL TUNING AND FIVE-SEED FORECASTS
# =============================================================================
def fit_tuning_model(model_name,Xtr,ytr,Xv,yv,hp,seed):
    set_seed(seed); tf.keras.backend.clear_session(); gc.collect()
    m=BUILDERS[model_name](Xtr.shape[1:],hp)
    hist=m.fit(Xtr,ytr,validation_data=(Xv,yv),epochs=MAX_EPOCHS,
               batch_size=BATCH_SIZE,shuffle=False,verbose=0,
               callbacks=[keras.callbacks.EarlyStopping(
                   monitor="val_loss",patience=EARLY_STOPPING_PATIENCE,
                   restore_best_weights=True,min_delta=1e-6)])
    best=float(np.nanmin(hist.history["val_loss"]))
    ep=int(np.nanargmin(hist.history["val_loss"])+1); npar=int(m.count_params())
    del m; tf.keras.backend.clear_session(); gc.collect()
    return best,ep,npar

def tune_sector(model_name,sd,target_col,seq_len):
    sdx,X,_=prepare_sector_matrix(sd,NEURAL_FEATURES_ALL,target_col,BOUND["Train_End"])
    Xq,yq,dq,_=make_sequences(sdx,X,target_col,seq_len)
    tr=dq<=BOUND["Train_End"]
    va=(dq>=BOUND["Validation_Start"])&(dq<=BOUND["Validation_End"])
    if tr.sum()<MIN_TRAIN_SEQUENCES or va.sum()<MIN_VAL_SEQUENCES: return None,[]
    rec=[]
    for hid,hp in enumerate(HPGRIDS[model_name],1):
        row={"Model":model_name,"Sector":sd["Sector"].iloc[0],"Sequence_Length":seq_len,
             "Hyperparameter_ID":hid,"Hyperparameters":json.dumps(hp,sort_keys=True),
             "N_Train_Sequences":int(tr.sum()),"N_Validation_Sequences":int(va.sum()),
             "Scaler_Fit_End":BOUND["Train_End"]}
        try:
            v,ep,np_=fit_tuning_model(model_name,Xq[tr],yq[tr],Xq[va],yq[va],hp,TUNING_SEED)
            row.update({"Validation_QLIKE":v,"Best_Epoch":ep,"Trainable_Parameters":np_})
        except Exception as e:
            row.update({"Validation_QLIKE":np.nan,"Error":repr(e)})
        rec.append(row)
    valid=[r for r in rec if np.isfinite(r.get("Validation_QLIKE",np.nan))]
    return (min(valid,key=lambda r:r["Validation_QLIKE"]) if valid else None),rec

def refit_forecast_sector(model_name,sd,target_col,seq_len,best,seeds,tag=None):
    tag=tag or model_name
    hp=json.loads(best["Hyperparameters"]); epochs=max(1,int(best["Best_Epoch"]))
    sdx,X,_=prepare_sector_matrix(sd,NEURAL_FEATURES_ALL,target_col,BOUND["Validation_End"])
    Xq,yq,dq,_=make_sequences(sdx,X,target_col,seq_len)
    fit=dq<=BOUND["Validation_End"]; ev=dq>=BOUND["StackTrain_Start"]
    if fit.sum()<MIN_TRAIN_SEQUENCES or ev.sum()<5: return pd.DataFrame(),pd.DataFrame()
    prs=[]; dia=[]; sector=sd["Sector"].iloc[0]
    for seed in seeds:
        try:
            set_seed(seed); tf.keras.backend.clear_session(); gc.collect()
            m=BUILDERS[model_name](Xq.shape[1:],hp)
            m.fit(Xq[fit],yq[fit],epochs=epochs,batch_size=BATCH_SIZE,shuffle=False,verbose=0)
            p=m.predict(Xq[ev],batch_size=BATCH_SIZE,verbose=0).reshape(-1)
            p=np.clip(p,EPSILON,FORECAST_UPPER_CLIP)
            for d,y,f in zip(dq[ev],yq[ev],p):
                prs.append({"Date":pd.Timestamp(d),"Sector":sector,"Model":tag,"Seed":seed,
                            "Forecast":float(f),"Actual_Variance":float(y),"Sequence_Length":seq_len})
            dia.append({"Sector":sector,"Model":tag,"Seed":seed,"Sequence_Length":seq_len,
                        "Fit_Sequences":int(fit.sum()),"Forecast_Sequences":int(ev.sum()),
                        "Selected_Epochs":epochs,"Trainable_Parameters":int(m.count_params()),
                        "Observations_Per_Parameter":float(fit.sum()/max(m.count_params(),1)),
                        "Hyperparameters":json.dumps(hp,sort_keys=True),
                        "Scaler_Fit_End":BOUND["Validation_End"],"Status":"OK"})
            del m; tf.keras.backend.clear_session(); gc.collect()
        except Exception as e:
            dia.append({"Sector":sector,"Model":tag,"Seed":seed,"Status":"FAILED","Error":repr(e)})
    ps=pd.DataFrame(prs); dd=pd.DataFrame(dia)
    if ps.empty: return pd.DataFrame(),dd
    ens=(ps.groupby(["Date","Sector","Model"],as_index=False)
         .agg(Forecast=("Forecast","mean"),Forecast_Seed_SD=("Forecast","std"),
              N_Seeds=("Seed","nunique"),Actual_Variance=("Actual_Variance","first"),
              Sequence_Length=("Sequence_Length","first")))
    return ens,pd.concat([dd,ps.assign(Record_Type="PerSeedForecast")],
                         ignore_index=True,sort=False)

def run_neural(
    model_name,
    target_col="Actual_Variance",
    seq_len=PRIMARY_SEQUENCE_LENGTH,
    stage="primary",
):
    """
    Tune/refit one model sector-by-sector with durable checkpoints.

    After each sector finishes, its ensemble, tuning rows and detailed
    seed-level forecasts are saved. A rerun loads completed sectors and starts
    at the first unfinished sector instead of repeating all 11 sectors.
    """
    tunes, ens, details = [], [], []
    sectors = sorted(features["Sector"].unique())

    for sector in tqdm(sectors, desc=f"{model_name} tune+fit"):
        ck = _load_sector_checkpoint(stage, model_name, seq_len, sector)
        if ck is not None:
            e_ck, t_ck, d_ck = ck
            if not e_ck.empty:
                ens.append(e_ck)
            if not t_ck.empty:
                tunes.extend(t_ck.to_dict("records"))
            if not d_ck.empty:
                details.append(d_ck)
            tqdm.write(
                f"RESUME: {stage} | {model_name} | seq={seq_len} | "
                f"{sector} loaded from checkpoint."
            )
            continue

        sd = features[features["Sector"] == sector].copy()
        best, recs = tune_sector(model_name, sd, target_col, seq_len)
        tdf = pd.DataFrame(recs)

        if best is None:
            edf = pd.DataFrame()
            ddf = pd.DataFrame([{
                "Sector": sector,
                "Model": model_name,
                "Sequence_Length": seq_len,
                "Status": "NO_VALID_TUNING_RESULT",
            }])
        else:
            edf, ddf = refit_forecast_sector(
                model_name, sd, target_col, seq_len, best, NEURAL_SEEDS
            )

        _write_sector_checkpoint(
            stage, model_name, seq_len, sector, edf, tdf, ddf
        )

        if not edf.empty:
            ens.append(edf)
        if not tdf.empty:
            tunes.extend(tdf.to_dict("records"))
        if not ddf.empty:
            details.append(ddf)

        # Release graphs between sectors.
        tf.keras.backend.clear_session()
        gc.collect()

    eall = pd.concat(ens, ignore_index=True) if ens else pd.DataFrame()
    tall = pd.DataFrame(tunes)
    dall = (
        pd.concat(details, ignore_index=True, sort=False)
        if details else pd.DataFrame()
    )
    return eall, tall, dall

def _complete_primary_neural_from_existing(model_name):
    """v5.09 never reuses earlier primary neural results."""
    return None


print("\n"+"="*100); print("NEURAL MODELS: GRU, TRANSFORMER, PATCHTST"); print("="*100)
neural={}; tunelist=[]; detaillist=[]
for mn in ["GRU","Transformer","PatchTST"]:
    resumed = _complete_primary_neural_from_existing(mn)
    if resumed is not None:
        e,t,d = resumed
    else:
        e,t,d = run_neural(
            mn,
            target_col="Actual_Variance",
            seq_len=PRIMARY_SEQUENCE_LENGTH,
            stage="primary",
        )
    if e.empty:
        raise RuntimeError(f"No forecasts produced for {mn}")
    neural[mn]=e
    tunelist.append(t)
    detaillist.append(d)
    save_csv(e,f"04_{mn}_Ensemble_Forecasts.csv")

tuning=pd.concat(tunelist,ignore_index=True,sort=False)
details=pd.concat(detaillist,ignore_index=True,sort=False)
save_csv(tuning,"04A_Neural_Hyperparameter_Tuning.csv")
save_csv(details,"04B_Neural_Seed_Parameter_Diagnostics.csv")

# Per-seed stability.
ps=details[details.get("Record_Type",pd.Series(index=details.index,dtype=object)).eq("PerSeedForecast")].copy()
sr=[]
if not ps.empty:
    for (m,s,seed),g in ps.groupby(["Model","Sector","Seed"]):
        sr.append({"Model":m,"Sector":s,"Seed":seed,"N":len(g),
                   "QLIKE":float(np.mean(qlike(g["Actual_Variance"],g["Forecast"]))),
                   "RMSE":float(np.sqrt(np.mean((g["Actual_Variance"]-g["Forecast"])**2))),
                   "MAE":float(np.mean(np.abs(g["Actual_Variance"]-g["Forecast"])))})
seedres=pd.DataFrame(sr)
save_csv(seedres,"04C_Neural_Seed_Stability.csv")
if not seedres.empty:
    save_csv(seedres.groupby(["Model","Sector"],as_index=False).agg(
        Mean_QLIKE=("QLIKE","mean"),SD_QLIKE=("QLIKE","std"),
        Min_QLIKE=("QLIKE","min"),Max_QLIKE=("QLIKE","max"),N_Seeds=("Seed","nunique")),
        "04D_Neural_Seed_Stability_Summary.csv")

# =============================================================================
# 9. COMMON BASELINE PANEL AND COVERAGE
# =============================================================================
panel=econ.copy()
panel=panel.merge(features[["Date","Sector","Volume_Return_Lag1"]],
                  on=["Date","Sector"],how="left",validate="one_to_one")
for mn in ["GRU","Transformer","PatchTST"]:
    ndf=neural[mn][["Date","Sector","Forecast","Forecast_Seed_SD","N_Seeds"]].rename(
        columns={"Forecast":f"{mn}_Forecast","Forecast_Seed_SD":f"{mn}_Seed_SD",
                 "N_Seeds":f"{mn}_N_Seeds"})
    panel=panel.merge(ndf,on=["Date","Sector"],how="left",validate="one_to_one")
panel["Evaluation_Split"]=panel["Date"].map(split_label)
panel_eval=panel[panel["Evaluation_Split"].isin(["StackTrain","FinalTest"])].copy()

for c in list(R_FORECAST_COLUMNS.values())+["GRU_Forecast","Transformer_Forecast","PatchTST_Forecast"]:
    panel_eval[c]=pd.to_numeric(panel_eval[c],errors="coerce")
    panel_eval.loc[panel_eval[c].notna(),c]=panel_eval.loc[panel_eval[c].notna(),c].clip(
        lower=EPSILON,upper=FORECAST_UPPER_CLIP)

panel_eval["FC_TGARCH_X"]=panel_eval["TGARCH_X_Forecast"]
panel_eval["FC_Transformer"]=panel_eval["Transformer_Forecast"]

ca=[]
for c in list(R_FORECAST_COLUMNS.values())+["GRU_Forecast","Transformer_Forecast","PatchTST_Forecast"]:
    for spn in ["StackTrain","FinalTest"]:
        g=panel_eval[panel_eval["Evaluation_Split"]==spn]
        ca.append({"Forecast_Column":c,"Split":spn,"N_Available":int(g[c].notna().sum()),
                   "N_Total":len(g),"Coverage":float(g[c].notna().mean()),
                   "Unique_Dates":g.loc[g[c].notna(),"Date"].nunique(),
                   "Unique_Sectors":g.loc[g[c].notna(),"Sector"].nunique()})
save_csv(pd.DataFrame(ca),"05_Forecast_Coverage_Audit.csv")

# =============================================================================
# 10. STATIC COMBINATION: CV CONVEX, EQUAL AND UNCONSTRAINED ROBUSTNESS
# =============================================================================
def qmean(y,f): return float(np.nanmean(qlike(y,f)))

def optimise_weight(y,tg,tf_,bounds=(0.0,1.0)):
    y=np.asarray(y,float); tg=np.asarray(tg,float); tf_=np.asarray(tf_,float)
    m=np.isfinite(y)&np.isfinite(tg)&np.isfinite(tf_)
    if m.sum()<MIN_STACK_OBS: return .5,np.nan
    y=y[m]; tg=tg[m]; tf_=tf_[m]
    def obj(w):
        f=np.clip(w*tf_+(1-w)*tg,EPSILON,FORECAST_UPPER_CLIP)
        return qmean(y,f)
    r=minimize_scalar(obj,bounds=bounds,method="bounded",options={"xatol":1e-5})
    return (float(r.x),float(r.fun)) if r.success else (.5,obj(.5))

def convex_weight_cv(g):
    g=g.dropna(subset=["Actual_Variance","FC_TGARCH_X","FC_Transformer"]).sort_values("Date")
    if len(g)<4*MIN_STACK_OBS:
        w,l=optimise_weight(g["Actual_Variance"],g["FC_TGARCH_X"],g["FC_Transformer"])
        return w,np.nan,"full_stacktrain_minimum"
    dates=np.array(sorted(g["Date"].unique()))
    ns=min(5,max(2,len(dates)//20)); foldloss=[]
    for tri,vai in TimeSeriesSplit(n_splits=ns).split(dates):
        tr=g[g["Date"].isin(set(dates[tri]))]; va=g[g["Date"].isin(set(dates[vai]))]
        if len(tr)<MIN_STACK_OBS or len(va)<5: continue
        w,_=optimise_weight(tr["Actual_Variance"],tr["FC_TGARCH_X"],tr["FC_Transformer"])
        f=w*va["FC_Transformer"].values+(1-w)*va["FC_TGARCH_X"].values
        foldloss.append(qmean(va["Actual_Variance"],f))
    wf,_=optimise_weight(g["Actual_Variance"],g["FC_TGARCH_X"],g["FC_Transformer"])
    return wf,(float(np.mean(foldloss)) if foldloss else np.nan),"expanding_CV_then_full_stacktrain_fit"

sw=[]
for sector in sorted(panel_eval["Sector"].unique()):
    tr=panel_eval[(panel_eval["Sector"]==sector)&(panel_eval["Evaluation_Split"]=="StackTrain")]
    w,cv,meth=convex_weight_cv(tr)
    wu,lu=optimise_weight(tr["Actual_Variance"],tr["FC_TGARCH_X"],tr["FC_Transformer"],
                          bounds=UNCONSTRAINED_WEIGHT_BOUNDS)
    sw.append({"Sector":sector,"Weight_Transformer_Convex":w,"Weight_TGARCH_X_Convex":1-w,
               "CV_QLIKE":cv,"Method":meth,
               "Weight_Transformer_Unconstrained":wu,"Weight_TGARCH_X_Unconstrained":1-wu,
               "Unconstrained_StackTrain_QLIKE":lu})
sw=pd.DataFrame(sw); save_csv(sw,"06_Static_Combination_Weights.csv")
wmap=sw.set_index("Sector")["Weight_Transformer_Convex"].to_dict()
wumap=sw.set_index("Sector")["Weight_Transformer_Unconstrained"].to_dict()
panel_eval["Weight_Transformer_Static"]=panel_eval["Sector"].map(wmap)
panel_eval["FC_Combined_Static"]=(panel_eval["Weight_Transformer_Static"]*panel_eval["FC_Transformer"]+
                                  (1-panel_eval["Weight_Transformer_Static"])*panel_eval["FC_TGARCH_X"]).clip(EPSILON,FORECAST_UPPER_CLIP)
panel_eval["Weight_Transformer_Unconstrained"]=panel_eval["Sector"].map(wumap)
panel_eval["FC_Combined_Unconstrained"]=(panel_eval["Weight_Transformer_Unconstrained"]*panel_eval["FC_Transformer"]+
                                         (1-panel_eval["Weight_Transformer_Unconstrained"])*panel_eval["FC_TGARCH_X"]).clip(EPSILON,FORECAST_UPPER_CLIP)
panel_eval["FC_Combined_Equal"]=(.5*panel_eval["FC_Transformer"]+.5*panel_eval["FC_TGARCH_X"]).clip(EPSILON,FORECAST_UPPER_CLIP)

# =============================================================================
# 11. PRIMARY REGIME MODEL: ZERO-NEUTRAL STUDENT-t HMM WITH SOFT PROBABILITIES
# =============================================================================
print("\n"+"="*100)
print("PRIMARY REGIME MODEL: ZERO-NEUTRAL STUDENT-t HMM WITH SOFT PROBABILITIES")
print("="*100)

# -------------------------------------------------------------------------
# 11.1 Sector diagnostics before HMM fitting
# -------------------------------------------------------------------------
def _regime_input_diagnostics():
    firm_counts=mv.groupby("Sector")["Stock"].nunique().to_dict()
    rows=[]
    for sector,g in features.groupby("Sector"):
        gf=g[g["Date"]<=BOUND["Validation_End"]].sort_values("Date")
        r=pd.to_numeric(gf["Sector_Return"],errors="coerce").dropna()
        if r.empty:
            continue
        nz=~np.isclose(r.values,0.0,atol=HMM_ZERO_ATOL)
        rows.append({
            "Sector":sector,
            "N_Firms":int(firm_counts.get(sector,0)),
            "Single_Firm_Sector":bool(firm_counts.get(sector,0)==1),
            "Fit_Observations":int(len(r)),
            "Zero_Return_Share":float(np.mean(~nz)),
            "Nonzero_Observations":int(np.sum(nz)),
            "Unique_Return_Values":int(r.nunique()),
            "Return_SD":float(r.std(ddof=1)),
            "Return_Skewness":float(stats.skew(r.values,bias=False)),
            "Return_Excess_Kurtosis":float(stats.kurtosis(r.values,fisher=True,bias=False)),
            "Max_Absolute_Return":float(np.max(np.abs(r.values))),
        })
    return pd.DataFrame(rows)

regime_input_diag=_regime_input_diagnostics()
save_csv(regime_input_diag,"06A_Regime_Input_Diagnostics.csv")

# -------------------------------------------------------------------------
# 11.2 Fast two-state HMM likelihood/filter
# -------------------------------------------------------------------------
@njit
def _logsum2_numba(a,b):
    m=a if a>b else b
    return m+math.log(math.exp(a-m)+math.exp(b-m))

@njit
def _logistic_numba(x):
    if x>=0:
        z=math.exp(-x)
        return 1.0/(1.0+z)
    z=math.exp(x)
    return z/(1.0+z)

@njit
def _student_logpdf_std(x,scale,nu):
    c=(math.lgamma((nu+1.0)/2.0)-math.lgamma(nu/2.0)
       -0.5*math.log(nu*math.pi)-math.log(scale))
    q=(x/scale)*(x/scale)/nu
    return c-0.5*(nu+1.0)*math.log1p(q)

@njit
def _gaussian_logpdf_std(x,scale):
    return -0.5*math.log(2.0*math.pi)-math.log(scale)-0.5*(x/scale)*(x/scale)

@njit
def _hmm2_student_nll(theta,z,zero,zero_neutral):
    p00=_logistic_numba(theta[0])
    p11=_logistic_numba(theta[1])
    s0=math.exp(theta[2])
    s1=s0+math.exp(theta[3])
    nu=2.05+math.exp(theta[4])

    lp00=math.log(max(p00,1e-14))
    lp01=math.log(max(1.0-p00,1e-14))
    lp10=math.log(max(1.0-p11,1e-14))
    lp11=math.log(max(p11,1e-14))

    den=max(2.0-p00-p11,1e-14)
    pi0=max((1.0-p11)/den,1e-14)
    pi1=max((1.0-p00)/den,1e-14)
    ps=pi0+pi1
    pi0/=ps; pi1/=ps

    if zero_neutral and zero[0]:
        e0=0.0; e1=0.0
    else:
        e0=_student_logpdf_std(z[0],s0,nu)
        e1=_student_logpdf_std(z[0],s1,nu)

    a0=math.log(pi0)+e0
    a1=math.log(pi1)+e1
    ll=_logsum2_numba(a0,a1)
    a0-=ll; a1-=ll
    total=ll

    for t in range(1,len(z)):
        pr0=_logsum2_numba(a0+lp00,a1+lp10)
        pr1=_logsum2_numba(a0+lp01,a1+lp11)

        if zero_neutral and zero[t]:
            e0=0.0; e1=0.0
        else:
            e0=_student_logpdf_std(z[t],s0,nu)
            e1=_student_logpdf_std(z[t],s1,nu)

        a0=pr0+e0
        a1=pr1+e1
        lt=_logsum2_numba(a0,a1)
        a0-=lt; a1-=lt
        total+=lt

    if not math.isfinite(total):
        return 1e50
    return -total

@njit
def _hmm2_student_filter(theta,z,zero,zero_neutral):
    p00=_logistic_numba(theta[0])
    p11=_logistic_numba(theta[1])
    s0=math.exp(theta[2])
    s1=s0+math.exp(theta[3])
    nu=2.05+math.exp(theta[4])

    den=max(2.0-p00-p11,1e-14)
    f0=max((1.0-p11)/den,1e-14)
    f1=max((1.0-p00)/den,1e-14)
    ss=f0+f1; f0/=ss; f1/=ss

    out=np.empty((len(z),2))
    for t in range(len(z)):
        if t>0:
            n0=f0*p00+f1*(1.0-p11)
            n1=f0*(1.0-p00)+f1*p11
            f0=n0; f1=n1

        if not (zero_neutral and zero[t]):
            l0=_student_logpdf_std(z[t],s0,nu)
            l1=_student_logpdf_std(z[t],s1,nu)
            mx=l0 if l0>l1 else l1
            f0*=math.exp(l0-mx)
            f1*=math.exp(l1-mx)

        ss=f0+f1
        if ss<=0.0 or not math.isfinite(ss):
            f0=0.5; f1=0.5
        else:
            f0/=ss; f1/=ss
        out[t,0]=f0; out[t,1]=f1
    return out

@njit
def _hmm2_gaussian_nll(theta,z,zero,zero_neutral):
    p00=_logistic_numba(theta[0])
    p11=_logistic_numba(theta[1])
    s0=math.exp(theta[2])
    s1=s0+math.exp(theta[3])

    lp00=math.log(max(p00,1e-14))
    lp01=math.log(max(1.0-p00,1e-14))
    lp10=math.log(max(1.0-p11,1e-14))
    lp11=math.log(max(p11,1e-14))

    den=max(2.0-p00-p11,1e-14)
    pi0=max((1.0-p11)/den,1e-14)
    pi1=max((1.0-p00)/den,1e-14)
    ps=pi0+pi1; pi0/=ps; pi1/=ps

    if zero_neutral and zero[0]:
        e0=0.0; e1=0.0
    else:
        e0=_gaussian_logpdf_std(z[0],s0)
        e1=_gaussian_logpdf_std(z[0],s1)

    a0=math.log(pi0)+e0; a1=math.log(pi1)+e1
    ll=_logsum2_numba(a0,a1); a0-=ll; a1-=ll; total=ll

    for t in range(1,len(z)):
        pr0=_logsum2_numba(a0+lp00,a1+lp10)
        pr1=_logsum2_numba(a0+lp01,a1+lp11)
        if zero_neutral and zero[t]:
            e0=0.0; e1=0.0
        else:
            e0=_gaussian_logpdf_std(z[t],s0)
            e1=_gaussian_logpdf_std(z[t],s1)
        a0=pr0+e0; a1=pr1+e1
        lt=_logsum2_numba(a0,a1); a0-=lt; a1-=lt; total+=lt

    if not math.isfinite(total):
        return 1e50
    return -total

@njit
def _hmm2_gaussian_filter(theta,z,zero,zero_neutral):
    p00=_logistic_numba(theta[0])
    p11=_logistic_numba(theta[1])
    s0=math.exp(theta[2])
    s1=s0+math.exp(theta[3])

    den=max(2.0-p00-p11,1e-14)
    f0=max((1.0-p11)/den,1e-14)
    f1=max((1.0-p00)/den,1e-14)
    ss=f0+f1; f0/=ss; f1/=ss

    out=np.empty((len(z),2))
    for t in range(len(z)):
        if t>0:
            n0=f0*p00+f1*(1.0-p11)
            n1=f0*(1.0-p00)+f1*p11
            f0=n0; f1=n1
        if not (zero_neutral and zero[t]):
            l0=_gaussian_logpdf_std(z[t],s0)
            l1=_gaussian_logpdf_std(z[t],s1)
            mx=l0 if l0>l1 else l1
            f0*=math.exp(l0-mx); f1*=math.exp(l1-mx)
        ss=f0+f1
        if ss<=0.0 or not math.isfinite(ss):
            f0=0.5; f1=0.5
        else:
            f0/=ss; f1/=ss
        out[t,0]=f0; out[t,1]=f1
    return out

# Warm up JIT once.
_dummy_z=np.array([0.1,0.0,-0.2,0.4],dtype=np.float64)
_dummy_zero=np.array([False,True,False,False])
_=_hmm2_student_nll(np.array([2.,2.,np.log(.35),np.log(.8),np.log(5.95)]),
                    _dummy_z,_dummy_zero,True)
_=_hmm2_gaussian_nll(np.array([2.,2.,np.log(.35),np.log(.8)]),
                     _dummy_z,_dummy_zero,True)

def _state_diagnostics_from_probs(sd,fit_mask,probs):
    state=np.argmax(probs,axis=1)
    av=pd.to_numeric(sd["Actual_Variance"],errors="coerce").values.astype(float)
    sf=state[fit_mask]
    avf=av[fit_mask]
    shares=[float(np.mean(sf==j)) for j in [0,1]]
    means=[
        float(np.mean(avf[sf==j])) if np.any(sf==j) else np.nan
        for j in [0,1]
    ]
    ratio=means[1]/max(means[0],EPSILON) if np.all(np.isfinite(means)) else np.nan
    return state,shares,means,ratio

def _fit_hmm2_sector(sd,fit_end,distribution="student_t",zero_neutral=True,
                     nstarts=HMM_TWO_STATE_STARTS,stage="Final"):
    sd=sd.sort_values("Date").copy().reset_index(drop=True)
    sector=sd["Sector"].iloc[0]
    fit_mask=(sd["Date"]<=fit_end).values
    nfit=int(fit_mask.sum())
    y=pd.to_numeric(sd["Sector_Return"],errors="coerce").values.astype(float)

    if nfit<MIN_MARKOV_OBS:
        return None,pd.DataFrame(),{
            "Sector":sector,"Stage":stage,"Accepted":False,
            "Reason":f"Only {nfit} fitting observations"
        }
    if not np.all(np.isfinite(y[fit_mask])):
        return None,pd.DataFrame(),{
            "Sector":sector,"Stage":stage,"Accepted":False,
            "Reason":"Non-finite fitting returns"
        }

    zero_fit=np.isclose(y[fit_mask],0.0,atol=HMM_ZERO_ATOL)
    base_mask=(~zero_fit) if zero_neutral else np.ones(nfit,dtype=bool)
    if int(base_mask.sum())<MIN_MARKOV_OBS//2:
        return None,pd.DataFrame(),{
            "Sector":sector,"Stage":stage,"Accepted":False,
            "Reason":"Too few non-zero returns for stable standardisation"
        }

    mu=float(np.mean(y[fit_mask][base_mask]))
    sig=float(np.std(y[fit_mask][base_mask],ddof=1))
    if not np.isfinite(sig) or sig<=EPSILON:
        return None,pd.DataFrame(),{
            "Sector":sector,"Stage":stage,"Accepted":False,
            "Reason":"Near-zero standard deviation"
        }

    z=(y-mu)/sig
    zfit=z[fit_mask]
    zero_all=np.isclose(y,0.0,atol=HMM_ZERO_ATOL)
    zero_arg=zero_fit.astype(np.bool_)

    # Deterministic multi-starts.
    seed=(sum(map(ord,str(sector)))+1009*len(str(stage))
          +(17 if distribution=="student_t" else 31)
          +(47 if zero_neutral else 59))
    rng=np.random.default_rng(seed)

    attempts=[]
    candidates=[]
    for j in range(int(nstarts)):
        persistence=np.array([2.0,2.0])+rng.normal(0,0.65,2)
        low=max(0.10,0.35*np.exp(rng.normal(0,0.30)))
        high=max(low+0.10,1.15*np.exp(rng.normal(0,0.30)))
        if distribution=="student_t":
            nu0=float(np.clip(6.0*np.exp(rng.normal(0,0.25)),2.6,30.0))
            x0=np.array([
                persistence[0],persistence[1],
                np.log(low),np.log(max(high-low,0.05)),
                np.log(max(nu0-2.05,0.05))
            ])
            bounds=[(-8,8),(-8,8),(-8,3),(-8,4),(-6,8)]
            obj=lambda th:_hmm2_student_nll(
                np.asarray(th,dtype=np.float64),zfit,zero_arg,zero_neutral)
        else:
            x0=np.array([
                persistence[0],persistence[1],
                np.log(low),np.log(max(high-low,0.05))
            ])
            bounds=[(-8,8),(-8,8),(-8,3),(-8,4)]
            obj=lambda th:_hmm2_gaussian_nll(
                np.asarray(th,dtype=np.float64),zfit,zero_arg,zero_neutral)

        try:
            res=minimize(obj,x0,method="L-BFGS-B",bounds=bounds,
                         options={"maxiter":HMM_MAXITER_2STATE,
                                  "ftol":1e-11,"gtol":1e-7})
            if distribution=="student_t":
                probs=_hmm2_student_filter(
                    np.asarray(res.x,dtype=np.float64),z,
                    zero_all.astype(np.bool_),zero_neutral)
                p00=float(expit(res.x[0])); p11=float(expit(res.x[1]))
                s0=float(np.exp(res.x[2]))
                s1=float(s0+np.exp(res.x[3]))
                nu=float(2.05+np.exp(res.x[4]))
            else:
                probs=_hmm2_gaussian_filter(
                    np.asarray(res.x,dtype=np.float64),z,
                    zero_all.astype(np.bool_),zero_neutral)
                p00=float(expit(res.x[0])); p11=float(expit(res.x[1]))
                s0=float(np.exp(res.x[2]))
                s1=float(s0+np.exp(res.x[3]))
                nu=np.nan

            state,shares,means,vr=_state_diagnostics_from_probs(sd,fit_mask,probs)
            transition_ok=bool(
                min(p00,1-p00,p11,1-p11)>=HMM_TRANSITION_EPS)
            occupancy_ok=bool(min(shares)>=HMM_MIN_STATE_SHARE)
            separation_ok=bool(np.isfinite(vr) and vr>=HMM_MIN_VARIANCE_RATIO)
            accepted=bool(res.success and np.isfinite(res.fun)
                          and transition_ok and occupancy_ok and separation_ok)

            row={
                "Sector":sector,"Stage":stage,"Distribution":distribution,
                "Zero_Neutral":zero_neutral,"Start":j+1,
                "Converged":bool(res.success),"Accepted":accepted,
                "NegLogLikelihood_StatePart":float(res.fun),
                "P_LL":p00,"P_HH":p11,
                "Sigma_Low_StdUnits":s0,"Sigma_High_StdUnits":s1,
                "Nu":nu,"Low_State_Share":shares[0],
                "High_State_Share":shares[1],
                "Variance_Ratio_High_to_Low":vr,
                "Transition_OK":transition_ok,
                "Occupancy_OK":occupancy_ok,
                "Variance_Separation_OK":separation_ok,
                "Message":str(res.message)
            }
            attempts.append(row)
            if accepted:
                candidates.append((float(res.fun),res,probs,row))
        except Exception as ex:
            attempts.append({
                "Sector":sector,"Stage":stage,"Distribution":distribution,
                "Zero_Neutral":zero_neutral,"Start":j+1,
                "Converged":False,"Accepted":False,"Error":repr(ex)
            })

    attempts_df=pd.DataFrame(attempts)
    if not candidates:
        return None,attempts_df,{
            "Sector":sector,"Stage":stage,"Distribution":distribution,
            "Zero_Neutral":zero_neutral,"Accepted":False,
            "Reason":"No multi-start candidate passed convergence, occupancy, "
                     "variance-separation and transition checks"
        }

    _,res,probs,brow=min(candidates,key=lambda x:x[0])
    state=np.argmax(probs,axis=1).astype(int)

    pzero=float(np.mean(zero_fit)) if zero_neutral else np.nan
    ll_state=-float(res.fun)
    if zero_neutral:
        n0=int(zero_fit.sum()); n1=nfit-n0
        ll_hurdle=(ll_state+n0*np.log(max(pzero,1e-14))
                   +n1*np.log(max(1-pzero,1e-14)))
    else:
        ll_hurdle=ll_state

    method=("ZeroNeutral_" if zero_neutral else "Conventional_")+(
        "StudentT" if distribution=="student_t" else "Gaussian"
    )+"_HMM2_FilteredForward"

    out=pd.DataFrame({
        "Date":sd["Date"],"Sector":sector,
        "Regime_State":state,
        "Regime_Probability_High":probs[:,1],
        "Regime_Method":method,
        "Markov_Converged":True,
        "HMM_Model_Signature":HMM_MODEL_SIGNATURE if (
            distribution=="student_t" and zero_neutral) else method,
    })
    out["Forecast_Regime"]=out["Regime_State"].shift(1)
    out["Forecast_Probability_High"]=out["Regime_Probability_High"].shift(1)

    meta={
        "Sector":sector,"Stage":stage,"Distribution":distribution,
        "Zero_Neutral":zero_neutral,"Accepted":True,
        "Fit_Observations":nfit,"Fit_End":fit_end,
        "Zero_Return_Share":pzero,
        "Standardisation_Mean":mu,"Standardisation_SD":sig,
        "P_LL":brow["P_LL"],"P_HH":brow["P_HH"],
        "Sigma_Low_StdUnits":brow["Sigma_Low_StdUnits"],
        "Sigma_High_StdUnits":brow["Sigma_High_StdUnits"],
        "Nu":brow["Nu"],
        "Low_State_Share":brow["Low_State_Share"],
        "High_State_Share":brow["High_State_Share"],
        "Variance_Ratio_High_to_Low":brow["Variance_Ratio_High_to_Low"],
        "Rare_State_Warning":bool(min(
            brow["Low_State_Share"],brow["High_State_Share"]
        )<HMM_RARE_STATE_WARNING_SHARE),
        "LogLikelihood_StatePart":ll_state,
        "LogLikelihood_Including_Common_Hurdle":ll_hurdle,
        "Selected_Start":int(brow["Start"]),
        "Model_Signature":out["HMM_Model_Signature"].iloc[0],
    }
    return out,attempts_df,meta

# -------------------------------------------------------------------------
# 11.3 Leakage-safe primary HMM: tuning fit and final fit
# -------------------------------------------------------------------------
tune_frames=[]; final_frames=[]
tune_attempts=[]; final_attempts=[]
tune_meta=[]; final_meta=[]
failed_tune=[]; failed_final=[]

for sector in tqdm(sorted(features["Sector"].unique()),desc="Primary HMM tuning-fit"):
    sd=features[features["Sector"]==sector].copy()
    o,a,m=_fit_hmm2_sector(
        sd,BOUND["Train_End"],distribution="student_t",
        zero_neutral=True,nstarts=max(4,HMM_TWO_STATE_STARTS-2),
        stage="TuningFit_Through_TrainEnd")
    tune_attempts.append(a); tune_meta.append(m)
    if o is None:
        failed_tune.append(sector)
    else:
        tune_frames.append(o)

for sector in tqdm(sorted(features["Sector"].unique()),desc="Primary HMM final-fit"):
    sd=features[features["Sector"]==sector].copy()
    o,a,m=_fit_hmm2_sector(
        sd,BOUND["Validation_End"],distribution="student_t",
        zero_neutral=True,nstarts=HMM_TWO_STATE_STARTS,
        stage="FinalFit_Through_ValidationEnd")
    final_attempts.append(a); final_meta.append(m)
    if o is None:
        failed_final.append(sector)
    else:
        final_frames.append(o)

save_csv(pd.concat(tune_attempts,ignore_index=True,sort=False),
         "07A0_Primary_HMM_TuningFit_Attempts.csv")
save_csv(pd.DataFrame(tune_meta),"07A1_Primary_HMM_TuningFit_Meta.csv")
save_csv(pd.concat(final_attempts,ignore_index=True,sort=False),
         "07A2_Primary_HMM_FinalFit_Attempts.csv")
save_csv(pd.DataFrame(final_meta),"07A3_Primary_HMM_FinalFit_Meta.csv")

if failed_tune or failed_final:
    raise RuntimeError(
        "PRIMARY HMM STOP: no fallback is permitted. "
        f"Tuning-fit failures={failed_tune}; final-fit failures={failed_final}. "
        "Inspect 07A0-07A3 outputs before proceeding."
    )

regimes_tuning=pd.concat(tune_frames,ignore_index=True).sort_values(["Sector","Date"])
regimes=pd.concat(final_frames,ignore_index=True).sort_values(["Sector","Date"])

save_csv(regimes_tuning,"07_Primary_HMM_TuningFit_Classification.csv")
save_csv(regimes,"07_Primary_HMM_Final_Classification.csv")

# Soft-probability uncertainty summary. Entropy is highest at p=0.5 and low
# near 0/1; it helps document whether the HMM produces informative yet
# genuinely probabilistic state assessments.
prob_summary=[]
for sector,g in regimes.groupby("Sector"):
    pp=pd.to_numeric(g["Regime_Probability_High"],errors="coerce").dropna().clip(1e-12,1-1e-12)
    ent=-(pp*np.log(pp)+(1-pp)*np.log(1-pp))
    prob_summary.append({
        "Sector":sector,"N":len(pp),
        "Mean_P_High":float(pp.mean()),"SD_P_High":float(pp.std(ddof=1)),
        "P10_P_High":float(pp.quantile(.10)),"Median_P_High":float(pp.median()),
        "P90_P_High":float(pp.quantile(.90)),
        "Share_P_Below_0.10":float(np.mean(pp<.10)),
        "Share_P_Above_0.90":float(np.mean(pp>.90)),
        "Mean_Binary_Entropy":float(ent.mean())
    })
save_csv(pd.DataFrame(prob_summary),"07G_Soft_Regime_Probability_Summary.csv")

# -------------------------------------------------------------------------
# 11.4 Zero-return/state validity audit for the PRIMARY final HMM
# -------------------------------------------------------------------------
def _phi_from_2x2(a,b,c,d):
    den=math.sqrt(max((a+b)*(c+d)*(a+c)*(b+d),0.0))
    return float((a*d-b*c)/den) if den>0 else np.nan

def _zero_return_regime_audit(regime_df):
    firm_counts=mv.groupby("Sector")["Stock"].nunique().to_dict()
    day=features[["Date","Sector","Sector_Return","Actual_Variance"]].merge(
        regime_df[["Date","Sector","Regime_State","Regime_Probability_High"]],
        on=["Date","Sector"],how="inner",validate="one_to_one")
    day["Sector_Return"]=pd.to_numeric(day["Sector_Return"],errors="coerce")
    day["Zero_Return"]=(day["Sector_Return"].abs()<=HMM_ZERO_ATOL).astype(int)
    day["Low_State"]=(day["Regime_State"].astype(int)==0).astype(int)
    day["Audit_Sample"]=np.where(
        day["Date"]<=BOUND["Validation_End"],"PreStackTrainFit","PostFitFiltered")
    save_csv(day,"07D0_ZeroReturn_Regime_DayLevel_Audit.csv")

    rows=[]; cont=[]
    for sector,gs in day.groupby("Sector"):
        for sample_name,mask,decision in [
            ("PreStackTrainFit",gs["Date"]<=BOUND["Validation_End"],True),
            ("FullAvailable",pd.Series(True,index=gs.index),False)
        ]:
            g=gs.loc[mask].dropna(subset=["Sector_Return","Regime_State"]).copy()
            zero=g["Zero_Return"].astype(bool).values
            low=g["Low_State"].astype(bool).values
            a=int(np.sum(zero&low)); b=int(np.sum(zero&~low))
            c=int(np.sum(~zero&low)); d=int(np.sum(~zero&~low))
            n=a+b+c+d; nz=a+b; nl=a+c; nnz=c+d
            plz=a/nz if nz else np.nan
            pzl=a/nl if nl else np.nan
            plnz=c/nnz if nnz else np.nan
            phi=_phi_from_2x2(a,b,c,d)
            try:
                odds,pv=stats.fisher_exact([[a,b],[c,d]],alternative="two-sided")
                odds=float(odds); pv=float(pv)
            except Exception:
                odds=np.nan; pv=np.nan

            av=pd.to_numeric(g["Actual_Variance"],errors="coerce")
            lowv=av[g["Low_State"].eq(1)]
            highv=av[g["Low_State"].eq(0)]
            ml=float(lowv.mean()) if len(lowv) else np.nan
            mh=float(highv.mean()) if len(highv) else np.nan
            vr=mh/max(ml,EPSILON) if np.isfinite(ml) and np.isfinite(mh) else np.nan

            x=g["Regime_State"].astype(int).values
            if len(x)>=2:
                n00=np.sum((x[:-1]==0)&(x[1:]==0)); n01=np.sum((x[:-1]==0)&(x[1:]==1))
                n10=np.sum((x[:-1]==1)&(x[1:]==0)); n11=np.sum((x[:-1]==1)&(x[1:]==1))
                pll=n00/max(n00+n01,1); phh=n11/max(n10+n11,1)
            else:
                pll=phh=np.nan

            strong=bool(
                (np.isfinite(phi) and abs(phi)>=ZERO_REGIME_PHI_THRESHOLD)
                or (not pd.isna(odds) and odds>=ZERO_REGIME_ODDS_RATIO_THRESHOLD))
            fail=bool(
                decision and np.isfinite(pzl) and np.isfinite(plz)
                and pzl>=ZERO_LOW_DOMINANCE_THRESHOLD
                and plz>=LOW_ZERO_CAPTURE_THRESHOLD and strong)
            warning=bool(
                decision and not fail and (
                    (np.isfinite(pzl) and pzl>=ZERO_LOW_WARNING_THRESHOLD)
                    or (np.isfinite(phi) and abs(phi)>=ZERO_REGIME_PHI_WARNING_THRESHOLD)))

            lowshare=nl/max(n,1)
            rows.append({
                "Sector":sector,"Audit_Sample":sample_name,"Decision_Sample":decision,
                "N_Firms":int(firm_counts.get(sector,0)),
                "Single_Firm_Sector":bool(firm_counts.get(sector,0)==1),
                "N_Observations":n,"Zero_Return_Share":nz/max(n,1),
                "Low_State_Share":lowshare,"High_State_Share":1-lowshare,
                "P_Low_Given_Zero":plz,"P_Zero_Given_Low":pzl,
                "P_Low_Given_Nonzero":plnz,
                "Phi_Coefficient":phi,"Fisher_Odds_Ratio":odds,
                "Fisher_Exact_P":pv,"Mean_ActualVariance_Low":ml,
                "Mean_ActualVariance_High":mh,
                "Variance_Ratio_High_to_Low":vr,
                "P_LL":pll,"P_HH":phh,
                "Expected_Low_Duration":1/max(1-pll,EPSILON) if np.isfinite(pll) else np.nan,
                "Expected_High_Duration":1/max(1-phh,EPSILON) if np.isfinite(phh) else np.nan,
                "Rare_State_Warning":bool(min(lowshare,1-lowshare)<HMM_RARE_STATE_WARNING_SHARE),
                "Validity_Warning":warning,"Validity_Fail":fail
            })
            for rv,rl in [(1,"Zero"),(0,"Nonzero")]:
                for sv,sl in [(1,"Low"),(0,"High")]:
                    cont.append({
                        "Sector":sector,"Audit_Sample":sample_name,
                        "Return_Group":rl,"Regime_State_Label":sl,
                        "Count":int(np.sum((g["Zero_Return"].values==rv)&
                                           (g["Low_State"].values==sv)))
                    })
    return pd.DataFrame(rows),pd.DataFrame(cont)

zero_regime_audit,zero_regime_contingency=_zero_return_regime_audit(regimes)
save_csv(zero_regime_audit,"07D_ZeroReturn_Regime_Validity_Audit.csv")
save_csv(zero_regime_contingency,"07D1_ZeroReturn_Regime_Contingency.csv")

decision_audit=zero_regime_audit[zero_regime_audit["Decision_Sample"].eq(True)].copy()
failed_zero_regime_sectors=sorted(
    decision_audit.loc[decision_audit["Validity_Fail"].eq(True),"Sector"].astype(str).unique())
warning_zero_regime_sectors=sorted(
    decision_audit.loc[decision_audit["Validity_Warning"].eq(True),"Sector"].astype(str).unique())

save_csv(pd.DataFrame([
    {"Metric":"Primary regime model","Value":HMM_MODEL_SIGNATURE},
    {"Metric":"Decision sample","Value":"PreStackTrainFit"},
    {"Metric":"Sectors failing zero-return dominance rule","Value":len(failed_zero_regime_sectors)},
    {"Metric":"Failed sectors","Value":"; ".join(failed_zero_regime_sectors)},
    {"Metric":"Warning-only sectors","Value":"; ".join(warning_zero_regime_sectors)},
    {"Metric":"Rare-state warning sectors","Value":"; ".join(
        decision_audit.loc[decision_audit["Rare_State_Warning"].eq(True),"Sector"].astype(str))},
]),"07D2_Regime_Validity_Preflight_Summary.csv")

print("\nPRIMARY HMM ZERO-RETURN / REGIME-VALIDITY PREFLIGHT")
print(decision_audit[[
    "Sector","N_Firms","Zero_Return_Share","Low_State_Share",
    "P_Low_Given_Zero","P_Zero_Given_Low","Phi_Coefficient",
    "Fisher_Odds_Ratio","Variance_Ratio_High_to_Low",
    "Rare_State_Warning","Validity_Warning","Validity_Fail"
]].to_string(index=False))

if (RUN_ZERO_RETURN_REGIME_PREFLIGHT and STOP_ON_ZERO_RETURN_REGIME_FAILURE
        and failed_zero_regime_sectors):
    raise RuntimeError(
        "PRIMARY REGIME-VALIDITY STOP: the zero-neutral Student-t HMM still "
        "shows zero-return dominance in: "+", ".join(failed_zero_regime_sectors)
        +". No probability-aware neural model was trained. Inspect 07D* outputs."
    )

# -------------------------------------------------------------------------
# 11.5 Robustness A: zero-neutral Gaussian HMM
# -------------------------------------------------------------------------
gauss_frames=[]; gauss_meta=[]; gauss_attempts=[]
if RUN_ZERO_NEUTRAL_GAUSSIAN_ROBUSTNESS:
    for sector in tqdm(sorted(features["Sector"].unique()),
                       desc="Zero-neutral Gaussian HMM robustness"):
        o,a,m=_fit_hmm2_sector(
            features[features["Sector"]==sector],BOUND["Validation_End"],
            distribution="gaussian",zero_neutral=True,nstarts=4,
            stage="Robustness_ZeroNeutral_Gaussian")
        gauss_attempts.append(a); gauss_meta.append(m)
        if o is not None: gauss_frames.append(o)
    save_csv(pd.concat(gauss_attempts,ignore_index=True,sort=False),
             "07E0_ZeroNeutral_Gaussian_Attempts.csv")
    save_csv(pd.DataFrame(gauss_meta),"07E1_ZeroNeutral_Gaussian_Meta.csv")
    gauss_states=pd.concat(gauss_frames,ignore_index=True) if gauss_frames else pd.DataFrame()
    save_csv(gauss_states,"07E2_ZeroNeutral_Gaussian_Classification.csv")
    if not gauss_states.empty:
        cmp=regimes[["Date","Sector","Regime_State"]].merge(
            gauss_states[["Date","Sector","Regime_State"]].rename(
                columns={"Regime_State":"Gaussian_State"}),
            on=["Date","Sector"],how="inner")
        agr=(cmp.assign(Agree=lambda d:d["Regime_State"].eq(d["Gaussian_State"]))
             .groupby("Sector",as_index=False)
             .agg(N=("Agree","size"),State_Agreement=("Agree","mean")))
        save_csv(agr,"07E3_Primary_vs_ZeroNeutralGaussian_Agreement.csv")

# -------------------------------------------------------------------------
# 11.6 Robustness B: conventional Student-t HMM (diagnostic)
# -------------------------------------------------------------------------
conv_frames=[]; conv_meta=[]; conv_attempts=[]
if RUN_CONVENTIONAL_STUDENTT_DIAGNOSTIC:
    for sector in tqdm(sorted(features["Sector"].unique()),
                       desc="Conventional Student-t HMM diagnostic"):
        o,a,m=_fit_hmm2_sector(
            features[features["Sector"]==sector],BOUND["Validation_End"],
            distribution="student_t",zero_neutral=False,nstarts=4,
            stage="Diagnostic_Conventional_StudentT")
        conv_attempts.append(a); conv_meta.append(m)
        if o is not None: conv_frames.append(o)
    save_csv(pd.concat(conv_attempts,ignore_index=True,sort=False),
             "07F0_Conventional_StudentT_Attempts.csv")
    save_csv(pd.DataFrame(conv_meta),"07F1_Conventional_StudentT_Meta.csv")
    conv_states=pd.concat(conv_frames,ignore_index=True) if conv_frames else pd.DataFrame()
    save_csv(conv_states,"07F2_Conventional_StudentT_Classification.csv")
    if not conv_states.empty:
        cmp=regimes[["Date","Sector","Regime_State"]].merge(
            conv_states[["Date","Sector","Regime_State"]].rename(
                columns={"Regime_State":"Conventional_StudentT_State"}),
            on=["Date","Sector"],how="inner")
        agr=(cmp.assign(
                Agree=lambda d:d["Regime_State"].eq(d["Conventional_StudentT_State"]))
             .groupby("Sector",as_index=False)
             .agg(N=("Agree","size"),State_Agreement=("Agree","mean")))
        save_csv(agr,"07F3_Primary_vs_ConventionalStudentT_Agreement.csv")

        # Explicitly document the zero-return/state association under the
        # conventional continuous Student-t HMM.
        zsrc=features[["Date","Sector","Sector_Return"]].merge(
            conv_states[["Date","Sector","Regime_State"]],
            on=["Date","Sector"],how="inner",validate="one_to_one")
        zsrc=zsrc[zsrc["Date"]<=BOUND["Validation_End"]].copy()
        zr=[]
        for sec,g in zsrc.groupby("Sector"):
            zero=np.isclose(
                pd.to_numeric(g["Sector_Return"],errors="coerce").values,
                0.0,atol=HMM_ZERO_ATOL)
            low=g["Regime_State"].astype(int).values==0
            a=int(np.sum(zero&low)); b=int(np.sum(zero&~low))
            c=int(np.sum(~zero&low)); d=int(np.sum(~zero&~low))
            phi=_phi_from_2x2(a,b,c,d)
            try:
                odds,pv=stats.fisher_exact([[a,b],[c,d]])
                odds=float(odds); pv=float(pv)
            except Exception:
                odds=np.nan; pv=np.nan
            zr.append({
                "Sector":sec,
                "P_Low_Given_Zero":a/max(a+b,1) if (a+b)>0 else np.nan,
                "P_Zero_Given_Low":a/max(a+c,1) if (a+c)>0 else np.nan,
                "P_Low_Given_Nonzero":c/max(c+d,1) if (c+d)>0 else np.nan,
                "Phi_Coefficient":phi,
                "Fisher_Odds_Ratio":odds,
                "Fisher_Exact_P":pv
            })
        save_csv(pd.DataFrame(zr),
                 "07F4_ConventionalStudentT_ZeroReturn_Association.csv")

# =============================================================================
# 12. PROBABILITY-AWARE TRANSFORMER: SOFT REGIME INFORMATION, FIVE SEEDS
# =============================================================================
print("\n"+"="*100)
print("PROBABILITY-AWARE TRANSFORMER: SOFT HMM PROBABILITY FEATURE")
print("="*100)

# Tuning probabilities come from an HMM fitted only through Train_End.
# Final probabilities come from an HMM refitted only through Validation_End.
# make_sequences uses rows [i-seq_len, ..., i-1] to forecast date i, so the
# latest HMM probability available to the neural forecast is p_{i-1}.
soft_tune=features.merge(
    regimes_tuning[["Date","Sector","Regime_Probability_High"]].rename(
        columns={"Regime_Probability_High":"Regime_Probability_High_Tuning"}),
    on=["Date","Sector"],how="left",validate="one_to_one")
soft_final=features.merge(
    regimes[["Date","Sector","Regime_Probability_High"]].rename(
        columns={"Regime_Probability_High":"Regime_Probability_High_Final"}),
    on=["Date","Sector"],how="left",validate="one_to_one")

def _prepare_soft_matrix(sd,prob_col,target_col,scaler_end_date):
    sd=sd.sort_values("Date").copy().reset_index(drop=True)
    train=sd[sd["Date"]<=scaler_end_date].copy()

    med=train[NEURAL_FEATURES].median(numeric_only=True).fillna(0.0)
    for c in NEURAL_FEATURES:
        sd[c]=sd[c].fillna(med[c])

    if sd[prob_col].isna().any():
        # Only protects any pre-sample edge; HMM filtering should normally
        # provide complete probabilities. 0.5 is neutral, not a state fallback.
        sd[prob_col]=sd[prob_col].fillna(0.5)

    feature_cols=NEURAL_FEATURES_ALL+[prob_col]
    Xraw=sd[feature_cols].astype(float).values
    smask=(sd["Date"]<=scaler_end_date).values
    mu=np.nanmean(Xraw[smask],axis=0)
    sig=np.nanstd(Xraw[smask],axis=0)
    mu=np.where(np.isfinite(mu),mu,0.0)
    sig=np.where(np.isfinite(sig)&(sig>EPSILON),sig,1.0)
    return sd,(Xraw-mu)/sig,np.column_stack([mu,sig]),feature_cols

def _tune_soft_transformer_sector(sd):
    sector=sd["Sector"].iloc[0]
    sdx,X,_,feature_cols=_prepare_soft_matrix(
        sd,"Regime_Probability_High_Tuning","Actual_Variance",BOUND["Train_End"])
    Xq,yq,dq,_=make_sequences(sdx,X,"Actual_Variance",PRIMARY_SEQUENCE_LENGTH)
    tr=dq<=BOUND["Train_End"]
    va=(dq>=BOUND["Validation_Start"])&(dq<=BOUND["Validation_End"])

    if tr.sum()<MIN_TRAIN_SEQUENCES or va.sum()<MIN_VAL_SEQUENCES:
        return None,pd.DataFrame([{
            "Model":"Probability-Aware Transformer","Sector":sector,
            "Status":"FAILED","Reason":"Insufficient chronological tuning sequences",
            "N_Train_Sequences":int(tr.sum()),"N_Validation_Sequences":int(va.sum())
        }])

    rec=[]
    for hid,hp in enumerate(TRANSFORMER_GRID,1):
        row={
            "Model":"Probability-Aware Transformer","Sector":sector,
            "Sequence_Length":PRIMARY_SEQUENCE_LENGTH,
            "Hyperparameter_ID":hid,
            "Hyperparameters":json.dumps(hp,sort_keys=True),
            "N_Train_Sequences":int(tr.sum()),
            "N_Validation_Sequences":int(va.sum()),
            "Scaler_Fit_End":BOUND["Train_End"],
            "HMM_Fit_End":BOUND["Train_End"],
            "Regime_Feature":"Filtered high-volatility probability in lagged sequence",
            "Feature_Count":len(feature_cols)
        }
        try:
            v,ep,np_=fit_tuning_model(
                "Transformer",Xq[tr],yq[tr],Xq[va],yq[va],hp,TUNING_SEED)
            row.update({
                "Validation_QLIKE":v,"Best_Epoch":ep,
                "Trainable_Parameters":np_,"Status":"OK"
            })
        except Exception as ex:
            row.update({"Validation_QLIKE":np.nan,"Status":"FAILED","Error":repr(ex)})
        rec.append(row)

    valid=[r for r in rec if np.isfinite(r.get("Validation_QLIKE",np.nan))]
    return (min(valid,key=lambda r:r["Validation_QLIKE"]) if valid else None,
            pd.DataFrame(rec))

def _fit_soft_transformer_sector(sd,best):
    sector=sd["Sector"].iloc[0]
    hp=json.loads(best["Hyperparameters"])
    epochs=max(1,int(best["Best_Epoch"]))

    sdx,X,_,feature_cols=_prepare_soft_matrix(
        sd,"Regime_Probability_High_Final","Actual_Variance",BOUND["Validation_End"])
    Xq,yq,dq,_=make_sequences(sdx,X,"Actual_Variance",PRIMARY_SEQUENCE_LENGTH)
    fit=dq<=BOUND["Validation_End"]
    ev=dq>=BOUND["StackTrain_Start"]

    if fit.sum()<MIN_TRAIN_SEQUENCES or ev.sum()<5:
        raise RuntimeError(
            f"{sector}: insufficient sequences for probability-aware Transformer.")

    seed_forecasts=[]; diag=[]
    for seed in NEURAL_SEEDS:
        try:
            set_seed(seed); tf.keras.backend.clear_session(); gc.collect()
            m=build_transformer(Xq.shape[1:],hp)
            m.fit(Xq[fit],yq[fit],epochs=epochs,batch_size=BATCH_SIZE,
                  shuffle=False,verbose=0)
            pred=m.predict(Xq[ev],batch_size=BATCH_SIZE,verbose=0).reshape(-1)
            pred=np.clip(pred,EPSILON,FORECAST_UPPER_CLIP)

            for d,yv,f in zip(dq[ev],yq[ev],pred):
                seed_forecasts.append({
                    "Date":pd.Timestamp(d),"Sector":sector,
                    "Model":"Probability-Aware Transformer",
                    "Seed":seed,"Forecast":float(f),
                    "Actual_Variance":float(yv),
                    "Sequence_Length":PRIMARY_SEQUENCE_LENGTH
                })
            diag.append({
                "Sector":sector,"Model":"Probability-Aware Transformer",
                "Seed":seed,"Status":"OK",
                "Fit_Sequences":int(fit.sum()),
                "Forecast_Sequences":int(ev.sum()),
                "Selected_Epochs":epochs,
                "Trainable_Parameters":int(m.count_params()),
                "Observations_Per_Parameter":float(
                    fit.sum()/max(int(m.count_params()),1)),
                "Hyperparameters":json.dumps(hp,sort_keys=True),
                "Scaler_Fit_End":BOUND["Validation_End"],
                "HMM_Fit_End":BOUND["Validation_End"],
                "Feature_Count":len(feature_cols)
            })
            del m; tf.keras.backend.clear_session(); gc.collect()
        except Exception as ex:
            diag.append({
                "Sector":sector,"Model":"Probability-Aware Transformer",
                "Seed":seed,"Status":"FAILED","Error":repr(ex)
            })

    ps=pd.DataFrame(seed_forecasts)
    dd=pd.DataFrame(diag)
    if ps.empty:
        return pd.DataFrame(),dd,pd.DataFrame()

    ens=(ps.groupby(["Date","Sector"],as_index=False)
         .agg(
             Regime_Transformer_Forecast=("Forecast","mean"),
             Regime_Transformer_Seed_SD=("Forecast","std"),
             Regime_Transformer_N_Seeds=("Seed","nunique"),
             Actual_Variance=("Actual_Variance","first")
         ))
    if pd.to_numeric(ens["Regime_Transformer_N_Seeds"],errors="coerce").min()<len(NEURAL_SEEDS):
        raise RuntimeError(
            f"{sector}: probability-aware Transformer did not complete all "
            f"{len(NEURAL_SEEDS)} seeds.")
    return ens,dd,ps

soft_ck_root=Path(CHECKPOINT_DIR)/(
    "probability_aware_transformer_"+_safe_slug(HMM_MODEL_SIGNATURE))
soft_ck_root.mkdir(parents=True,exist_ok=True)

soft_ens=[]; soft_tuning=[]; soft_details=[]; soft_seedrows=[]
for sector in tqdm(sorted(features["Sector"].unique()),
                   desc="Probability-aware Transformer"):
    slug=_safe_slug(sector)
    ep=soft_ck_root/f"{slug}_ensemble.csv"
    tp=soft_ck_root/f"{slug}_tuning.csv"
    dp=soft_ck_root/f"{slug}_diagnostics.csv"
    sp=soft_ck_root/f"{slug}_perseed.csv"
    jp=soft_ck_root/f"{slug}.done.json"

    loaded=False
    if all(x.exists() for x in [ep,tp,dp,sp,jp]):
        try:
            with open(jp,"r",encoding="utf-8") as fh:
                meta=json.load(fh)
            if meta.get("hmm_signature")==HMM_MODEL_SIGNATURE:
                ee=pd.read_csv(ep,low_memory=False)
                tt_=pd.read_csv(tp,low_memory=False)
                dd=pd.read_csv(dp,low_memory=False)
                ss=pd.read_csv(sp,low_memory=False)
                for df in [ee,ss]:
                    if "Date" in df: df["Date"]=parse_dates(df["Date"])
                if (not ee.empty and
                    pd.to_numeric(ee["Regime_Transformer_N_Seeds"],
                                  errors="coerce").min()>=len(NEURAL_SEEDS)):
                    soft_ens.append(ee); soft_tuning.append(tt_)
                    soft_details.append(dd); soft_seedrows.append(ss)
                    loaded=True
                    tqdm.write(f"RESUME: probability-aware Transformer | {sector}")
        except Exception:
            loaded=False
    if loaded:
        continue

    tune_sd=soft_tune[soft_tune["Sector"]==sector].copy()
    best,tune_df=_tune_soft_transformer_sector(tune_sd)
    if best is None:
        _atomic_csv_write(tune_df,tp,index=False)
        raise RuntimeError(
            f"{sector}: no valid probability-aware Transformer hyperparameter result.")

    final_sd=soft_final[soft_final["Sector"]==sector].copy()
    ee,dd,ss=_fit_soft_transformer_sector(final_sd,best)
    if ee.empty:
        raise RuntimeError(
            f"{sector}: no probability-aware Transformer forecasts produced.")

    _atomic_csv_write(ee,ep,index=False)
    _atomic_csv_write(tune_df,tp,index=False)
    _atomic_csv_write(dd,dp,index=False)
    _atomic_csv_write(ss,sp,index=False)
    tmp=jp.with_suffix(".json.tmp")
    with open(tmp,"w",encoding="utf-8") as fh:
        json.dump({
            "sector":sector,"hmm_signature":HMM_MODEL_SIGNATURE,
            "code_version":CODE_VERSION,"completed_at_unix":time.time()
        },fh,indent=2)
    tmp.replace(jp)

    soft_ens.append(ee); soft_tuning.append(tune_df)
    soft_details.append(dd); soft_seedrows.append(ss)

regtf=pd.concat(soft_ens,ignore_index=True)
regtune=pd.concat(soft_tuning,ignore_index=True,sort=False)
regdiag=pd.concat(soft_details,ignore_index=True,sort=False)
regseed=pd.concat(soft_seedrows,ignore_index=True,sort=False)

save_csv(regtf,"08_ProbabilityAware_Transformer_Forecasts.csv")
save_csv(regtune,"08A_ProbabilityAware_Transformer_Tuning.csv")
save_csv(regdiag,"08B_ProbabilityAware_Transformer_Diagnostics.csv")
save_csv(regseed,"08C_ProbabilityAware_Transformer_PerSeed_Forecasts.csv")

# Merge forecast-time regime information. p_{t-1} is used for a forecast
# dated t; the contemporaneous p_t is retained only for historical sequences.
panel_eval=panel_eval.merge(
    regimes[[
        "Date","Sector","Forecast_Regime","Forecast_Probability_High",
        "Regime_Method","Markov_Converged","HMM_Model_Signature"
    ]],
    on=["Date","Sector"],how="left",validate="one_to_one")

if panel_eval["Forecast_Probability_High"].isna().any():
    bad=panel_eval.loc[
        panel_eval["Forecast_Probability_High"].isna(),["Sector","Date"]]
    raise RuntimeError(
        f"Missing forecast-time HMM probabilities for {len(bad)} "
        "StackTrain/FinalTest sector-days.")
if not panel_eval["Forecast_Probability_High"].between(0,1,inclusive="both").all():
    raise RuntimeError("Forecast-time HMM probabilities fall outside [0,1].")

panel_eval=panel_eval.merge(
    regtf[[
        "Date","Sector","Regime_Transformer_Forecast",
        "Regime_Transformer_Seed_SD","Regime_Transformer_N_Seeds"
    ]],
    on=["Date","Sector"],how="left",validate="one_to_one")

missing_regime_tf=panel_eval["Regime_Transformer_Forecast"].isna()
if missing_regime_tf.any():
    bad=panel_eval.loc[missing_regime_tf,["Sector","Date"]]
    raise RuntimeError(
        "Probability-aware Transformer forecasts are missing for "
        f"{len(bad)} StackTrain/FinalTest sector-days. No baseline substitution "
        "was performed.")

panel_eval["Regime_Transformer_Fallback"]=False
panel_eval["FC_Transformer_Regime"]=panel_eval[
    "Regime_Transformer_Forecast"].clip(EPSILON,FORECAST_UPPER_CLIP)

# =============================================================================
# 13. PROBABILITY-AWARE TGARCH-X CALIBRATION AND FORECAST COMBINATION
# =============================================================================
def _weighted_calfactor(y,f,w):
    y=np.asarray(y,float); f=np.asarray(f,float); w=np.asarray(w,float)
    m=np.isfinite(y)&np.isfinite(f)&np.isfinite(w)&(f>EPSILON)&(w>=0)
    if not m.any() or np.sum(w[m])<=EPSILON:
        return np.nan,0.0
    ratio=y[m]/f[m]
    return float(np.average(ratio,weights=w[m])),float(np.sum(w[m]))

# Soft calibration factors are learned on StackTrain only and shrunk toward
# the unconditional StackTrain factor. No FinalTest outcomes enter calibration.
cal=[]; calmap={}
for sector in sorted(panel_eval["Sector"].unique()):
    st=panel_eval[
        (panel_eval["Sector"]==sector)&
        (panel_eval["Evaluation_Split"]=="StackTrain")
    ].dropna(subset=[
        "Actual_Variance","FC_TGARCH_X","Forecast_Probability_High"
    ]).copy()

    if len(st)<MIN_STACK_OBS:
        raise RuntimeError(
            f"{sector}: insufficient StackTrain observations for TGARCH-X calibration.")

    y=st["Actual_Variance"].values
    f=st["FC_TGARCH_X"].values
    p=st["Forecast_Probability_High"].clip(0,1).values

    ca,_=_weighted_calfactor(y,f,np.ones(len(st)))
    cl,nl=_weighted_calfactor(y,f,1-p)
    ch,nh=_weighted_calfactor(y,f,p)

    if not np.isfinite(ca):
        raise RuntimeError(f"{sector}: unconditional TGARCH-X calibration is undefined.")
    if not np.isfinite(cl): cl=ca
    if not np.isfinite(ch): ch=ca

    sl=nl/(nl+CALIBRATION_SHRINKAGE_K)
    sh=nh/(nh+CALIBRATION_SHRINKAGE_K)
    cls=float(np.clip(sl*cl+(1-sl)*ca,*CALIBRATION_FACTOR_BOUNDS))
    chs=float(np.clip(sh*ch+(1-sh)*ca,*CALIBRATION_FACTOR_BOUNDS))
    calmap[sector]=(cls,chs)

    cal.append({
        "Sector":sector,"N_StackTrain":len(st),
        "Unconditional_Factor":ca,
        "Raw_Low_ProbabilityWeighted_Factor":cl,
        "Raw_High_ProbabilityWeighted_Factor":ch,
        "Effective_Low_Weight":nl,"Effective_High_Weight":nh,
        "Low_Shrinkage_Weight":sl,"High_Shrinkage_Weight":sh,
        "Shrunk_Low_Factor":cls,"Shrunk_High_Factor":chs,
        "Method":"Probability-weighted StackTrain calibration with shrinkage"
    })

save_csv(pd.DataFrame(cal),"09_TGARCH_X_Probability_Calibration_Factors.csv")

def _calibration_for_row(sector,p):
    lo,hi=calmap[sector]
    pp=float(np.clip(p,0,1))
    return (1-pp)*lo+pp*hi

panel_eval["TGARCH_X_Calibration_Factor"]=[
    _calibration_for_row(s,p)
    for s,p in zip(panel_eval["Sector"],panel_eval["Forecast_Probability_High"])
]
panel_eval["FC_TGARCH_X_RegCal"]=(
    panel_eval["FC_TGARCH_X"]*panel_eval["TGARCH_X_Calibration_Factor"]
).clip(EPSILON,FORECAST_UPPER_CLIP)

# Probability-dependent convex combination:
#   w_t = logistic(a + b * p_{t-1})
#   f_t = w_t * Transformer_t + (1-w_t) * TGARCH-X_t
# a,b are estimated on StackTrain only.
def _fit_probability_combination(g):
    g=g.dropna(subset=[
        "Actual_Variance","FC_TGARCH_X_RegCal",
        "FC_Transformer_Regime","Forecast_Probability_High"
    ]).copy()
    if len(g)<MIN_STACK_OBS:
        raise RuntimeError("Insufficient StackTrain observations for probability combination.")

    y=g["Actual_Variance"].values.astype(float)
    tg=g["FC_TGARCH_X_RegCal"].values.astype(float)
    tf_=g["FC_Transformer_Regime"].values.astype(float)
    p=g["Forecast_Probability_High"].clip(0,1).values.astype(float)
    if np.nanstd(p)<1e-4:
        raise RuntimeError(
            "High-regime probability has negligible StackTrain variation; "
            "probability-dependent combination is unidentified.")

    # Nested constant convex combination for initialisation/comparison.
    wc,lc=optimise_weight(y,tg,tf_)
    wc=float(np.clip(wc,1e-5,1-1e-5))
    a0=float(np.log(wc/(1-wc)))

    def obj(ab):
        w=expit(ab[0]+ab[1]*p)
        fc=np.clip(w*tf_+(1-w)*tg,EPSILON,FORECAST_UPPER_CLIP)
        return qmean(y,fc)

    res=minimize(
        obj,np.array([a0,0.0]),method="L-BFGS-B",
        bounds=[REGIME_COMBINATION_A_BOUNDS,REGIME_COMBINATION_B_BOUNDS],
        options={"maxiter":2000,"ftol":1e-12}
    )
    if not res.success or not np.isfinite(res.fun):
        raise RuntimeError(f"Probability combination optimisation failed: {res.message}")

    a,b=map(float,res.x)
    w=expit(a+b*p)

    # Expanding chronological CV diagnostic. The final a,b above remain fit
    # on all StackTrain observations; CV is reported only to quantify weight
    # stability and guard against an apparently good in-sample slope.
    cv_losses=[]
    dates=np.array(sorted(pd.to_datetime(g["Date"].unique())))
    if len(dates)>=30:
        ns=min(5,max(2,len(dates)//20))
        for tri,vai in TimeSeriesSplit(n_splits=ns).split(dates):
            trm=g["Date"].isin(set(dates[tri])).values
            vam=g["Date"].isin(set(dates[vai])).values
            if trm.sum()<MIN_STACK_OBS or vam.sum()<5:
                continue
            yt=y[trm]; tgt=tg[trm]; tft=tf_[trm]; pt=p[trm]
            w0,_=optimise_weight(yt,tgt,tft)
            w0=float(np.clip(w0,1e-5,1-1e-5))
            aa0=float(np.log(w0/(1-w0)))
            def cvobj(ab):
                ww=expit(ab[0]+ab[1]*pt)
                ff=np.clip(ww*tft+(1-ww)*tgt,EPSILON,FORECAST_UPPER_CLIP)
                return qmean(yt,ff)
            rr=minimize(
                cvobj,np.array([aa0,0.0]),method="L-BFGS-B",
                bounds=[REGIME_COMBINATION_A_BOUNDS,REGIME_COMBINATION_B_BOUNDS],
                options={"maxiter":1000,"ftol":1e-11})
            if rr.success:
                wv=expit(rr.x[0]+rr.x[1]*p[vam])
                fv=np.clip(wv*tf_[vam]+(1-wv)*tg[vam],
                           EPSILON,FORECAST_UPPER_CLIP)
                cv_losses.append(qmean(y[vam],fv))

    return {
        "Intercept_a":a,"Slope_b_HighProbability":b,
        "StackTrain_QLIKE_ProbabilityAware":float(res.fun),
        "Chronological_CV_QLIKE":float(np.mean(cv_losses)) if cv_losses else np.nan,
        "Chronological_CV_Folds":len(cv_losses),
        "Nested_Constant_Weight_Transformer":wc,
        "Nested_Constant_QLIKE":float(lc),
        "Min_ProbabilityAware_Weight":float(np.min(w)),
        "Mean_ProbabilityAware_Weight":float(np.mean(w)),
        "Max_ProbabilityAware_Weight":float(np.max(w)),
        "N_StackTrain":len(g),
        "Optimiser":"L-BFGS-B",
        "Converged":True
    }

comb=[]; combmap={}
for sector in sorted(panel_eval["Sector"].unique()):
    st=panel_eval[
        (panel_eval["Sector"]==sector)&
        (panel_eval["Evaluation_Split"]=="StackTrain")
    ].copy()
    res=_fit_probability_combination(st)
    combmap[sector]=(res["Intercept_a"],res["Slope_b_HighProbability"],
                     res["Nested_Constant_Weight_Transformer"])
    comb.append({"Sector":sector,**res})

comb=pd.DataFrame(comb)
save_csv(comb,"10_ProbabilityAware_Combination_Parameters.csv")

weights=[]; nested=[]
for s,pv in zip(panel_eval["Sector"],panel_eval["Forecast_Probability_High"]):
    a,b,wc=combmap[s]
    weights.append(float(expit(a+b*float(np.clip(pv,0,1)))))
    nested.append(float(wc))

panel_eval["Weight_Transformer_Regime"]=weights
panel_eval["Weight_Transformer_Regime_NestedConstant"]=nested

panel_eval["FC_Combined_Regime"]=(
    panel_eval["Weight_Transformer_Regime"]*panel_eval["FC_Transformer_Regime"]
    +(1-panel_eval["Weight_Transformer_Regime"])*panel_eval["FC_TGARCH_X_RegCal"]
).clip(EPSILON,FORECAST_UPPER_CLIP)

# Nested/equal probability-component robustness forecasts.
panel_eval["FC_Combined_Regime_NestedConstant"]=(
    panel_eval["Weight_Transformer_Regime_NestedConstant"]*panel_eval["FC_Transformer_Regime"]
    +(1-panel_eval["Weight_Transformer_Regime_NestedConstant"])*panel_eval["FC_TGARCH_X_RegCal"]
).clip(EPSILON,FORECAST_UPPER_CLIP)
panel_eval["FC_Combined_Regime_Equal"]=(
    .5*panel_eval["FC_Transformer_Regime"]+.5*panel_eval["FC_TGARCH_X_RegCal"]
).clip(EPSILON,FORECAST_UPPER_CLIP)

save_csv(panel_eval,"11_Complete_Forecast_Panel_StackTrain_FinalTest.csv")

# =============================================================================
# 14. LOSS PANEL AND ACCURACY
# =============================================================================
ALL_MODEL_COLS={
    "Historical Variance":"Historical_Variance_Forecast",
    "EWMA":"EWMA_Forecast","GARCH":"GARCH_Forecast","GJR-GARCH":"GJR_GARCH_Forecast",
    "TGARCH":"TGARCH_Forecast_NoX","Baseline TGARCH-X":"FC_TGARCH_X",
    "GRU":"GRU_Forecast","Baseline Transformer":"FC_Transformer","PatchTST":"PatchTST_Forecast",
    "Equal Combined Forecast":"FC_Combined_Equal",
    "Unconstrained Combined Forecast":"FC_Combined_Unconstrained",
    "Static Combined Forecast":"FC_Combined_Static",
    "Regime-Calibrated TGARCH-X":"FC_TGARCH_X_RegCal",
    "Regime-Aware Transformer":"FC_Transformer_Regime",
    "Regime-Aware Combined Forecast":"FC_Combined_Regime",
    "Regime-Aware Nested Constant Combined Forecast":"FC_Combined_Regime_NestedConstant",
    "Regime-Aware Equal Combined Forecast":"FC_Combined_Regime_Equal",
}
def build_loss_long(df,actual_col="Actual_Variance",robustness_label="Primary squared-return target",
                    robustness_type="primary"):
    rec=[]
    keep=["Date","Sector","Evaluation_Split","Sector_Return","Actual_Variance","Parkinson_Variance",
          "Volume_Return_Lag1","Forecast_Regime","Forecast_Probability_High",
          "Markov_Converged","Regime_Method"]+list(WEIGHT_SCHEMES.values())
    for model,col in ALL_MODEL_COLS.items():
        if col not in df: continue
        x=df[keep+[col]].rename(columns={col:"Forecast"}).copy()
        x["Model"]=model; x["Evaluation_Target"]=actual_col
        x["Actual"]=pd.to_numeric(x[actual_col],errors="coerce")
        x["Forecast"]=pd.to_numeric(x["Forecast"],errors="coerce")
        x=x.dropna(subset=["Actual","Forecast"]).copy()
        x["Forecast"]=x["Forecast"].clip(EPSILON,FORECAST_UPPER_CLIP)
        x["QLIKE"]=qlike(x["Actual"],x["Forecast"])
        x["Squared_Error"]=(x["Actual"]-x["Forecast"])**2
        x["Absolute_Error"]=np.abs(x["Actual"]-x["Forecast"])
        x["Robustness_Label"]=robustness_label; x["Robustness_Type"]=robustness_type
        rec.append(x)
    return pd.concat(rec,ignore_index=True) if rec else pd.DataFrame()

loss_long=build_loss_long(panel_eval)
save_csv(loss_long,"12_Losses_Long_All_Models.csv")

def accuracy(ll,weight_scheme,split="FinalTest",state=None):
    wcol=WEIGHT_SCHEMES[weight_scheme]; x=ll[ll["Evaluation_Split"]==split].copy()
    if state is not None: x=x[x["Forecast_Regime"]==state]
    rec=[]
    for model,g in x.groupby("Model"):
        dr=[]
        for date,gd in g.groupby("Date"):
            w=gd[wcol].astype(float).values
            if np.nansum(w)<=0: continue
            wn=w/np.nansum(w)
            dr.append({"Date":date,
                       "QLIKE":weighted_mean_safe(gd["QLIKE"],wn),
                       "MSE":weighted_mean_safe(gd["Squared_Error"],wn),
                       "MAE":weighted_mean_safe(gd["Absolute_Error"],wn),
                       "N_Sectors":gd["Sector"].nunique(),
                       "Raw_Weight_Coverage":float(np.nansum(w))})
        d=pd.DataFrame(dr)
        if d.empty: continue
        rec.append({"Weight_Scheme":weight_scheme,"Split":split,
                    "State":"All" if state is None else ("High" if state==1 else "Low"),
                    "Model":model,"QLIKE":d["QLIKE"].mean(),"RMSE":np.sqrt(d["MSE"].mean()),
                    "MAE":d["MAE"].mean(),"N_Daily_Aggregates":d["Date"].nunique(),
                    "Mean_Sectors_Per_Day":d["N_Sectors"].mean(),
                    "Mean_Raw_Weight_Coverage":d["Raw_Weight_Coverage"].mean()})
    return pd.DataFrame(rec)

af=[]
for ws in WEIGHT_SCHEMES:
    for st in [None,0,1]: af.append(accuracy(loss_long,ws,"FinalTest",st))
acc=pd.concat(af,ignore_index=True)
acc["QLIKE_Rank"]=acc.groupby(["Weight_Scheme","State"])["QLIKE"].rank(method="min")
save_csv(acc,"13_Accuracy_Overall_State_WeightSchemes.csv")

sector_acc=(loss_long[loss_long["Evaluation_Split"]=="FinalTest"]
            .groupby(["Sector","Model"],as_index=False)
            .agg(QLIKE=("QLIKE","mean"),MSE=("Squared_Error","mean"),
                 MAE=("Absolute_Error","mean"),N=("Date","nunique")))
sector_acc["RMSE"]=np.sqrt(sector_acc["MSE"])
sector_acc["QLIKE_Rank"]=sector_acc.groupby("Sector")["QLIKE"].rank(method="min")
save_csv(sector_acc,"13A_Sector_Level_Accuracy_Ranks.csv")

state_cov=(loss_long[(loss_long["Evaluation_Split"]=="FinalTest")&
                     (loss_long["Model"]=="Baseline TGARCH-X")]
           .groupby(["Date","Forecast_Regime"],as_index=False)
           .agg(Sector_Days=("Sector","nunique"),
                MarketCap_Weight_Coverage=("Market_Weight_Daily","sum"),
                Equal_Weight_Coverage=("Equal_Weight_Daily","sum"),
                Capped_Weight_Coverage=("Capped_Market_Weight_Daily","sum")))
save_csv(state_cov,"13B_FinalTest_State_Date_Coverage.csv")

# Soft-probability performance bins complement the conventional hard-state
# low/high tables. Bins are fixed ex ante rather than estimated from FinalTest.
prob_bin_records=[]
pl=loss_long[loss_long["Evaluation_Split"]=="FinalTest"].copy()
pl["Probability_Bin"]=pd.cut(
    pl["Forecast_Probability_High"],
    bins=[-1e-12,1/3,2/3,1+1e-12],
    labels=["Low P(High)","Intermediate P(High)","High P(High)"],
    include_lowest=True)
for (pb,model),g in pl.dropna(subset=["Probability_Bin"]).groupby(
        ["Probability_Bin","Model"],observed=True):
    vals=[]
    for date,gd in g.groupby("Date"):
        w=gd["Market_Weight_Daily"].astype(float).values
        if np.nansum(w)<=0:
            continue
        vals.append(weighted_mean_safe(gd["QLIKE"],w/np.nansum(w)))
    prob_bin_records.append({
        "Probability_Bin":str(pb),"Model":model,
        "QLIKE":float(np.mean(vals)) if vals else np.nan,
        "N_Dates":g["Date"].nunique(),"N_SectorDays":len(g),
        "Mean_P_High":float(g["Forecast_Probability_High"].mean())
    })
save_csv(pd.DataFrame(prob_bin_records),"13C_Accuracy_Fixed_Probability_Bins.csv")

# =============================================================================
# 15. DAILY WEIGHTED LOSSES
# =============================================================================
def daily_losses_fn(ll,weight_scheme,state=None,split="FinalTest"):
    wcol=WEIGHT_SCHEMES[weight_scheme]; x=ll[ll["Evaluation_Split"]==split].copy()
    if state is not None: x=x[x["Forecast_Regime"]==state]
    rr=[]
    for (date,model),g in x.groupby(["Date","Model"]):
        w=g[wcol].astype(float).values
        if np.nansum(w)<=0: continue
        wn=w/np.nansum(w)
        rr.append({"Date":date,"Model":model,"QLIKE":weighted_mean_safe(g["QLIKE"],wn),
                   "MSE":weighted_mean_safe(g["Squared_Error"],wn),
                   "MAE":weighted_mean_safe(g["Absolute_Error"],wn),
                   "N_Sectors":g["Sector"].nunique(),"Raw_Weight_Coverage":float(np.nansum(w)),
                   "State":"All" if state is None else ("High" if state==1 else "Low"),
                   "Weight_Scheme":weight_scheme})
    return pd.DataFrame(rr)
dl=[]
for ws in WEIGHT_SCHEMES:
    for st in [None,0,1]: dl.append(daily_losses_fn(loss_long,ws,st))
daily_losses=pd.concat(dl,ignore_index=True)
save_csv(daily_losses,"14_Daily_Weighted_Losses.csv")

# =============================================================================
# 16. DM-HAC + HARVEY-LEYBOURNE-NEWBOLD + HOLM
# =============================================================================
def dm_hac_hln(l1,l2,h=1,bandwidth=None):
    d=np.asarray(l1,float)-np.asarray(l2,float); d=d[np.isfinite(d)]; n=len(d)
    if n<20: return np.nan,np.nan,np.nan,n,np.nan
    if bandwidth is None: bandwidth=max(h-1,int(np.floor(4*(n/100)**(2/9))))
    res=sm.OLS(d,np.ones((n,1))).fit(cov_type="HAC",
            cov_kwds={"maxlags":int(bandwidth),"use_correction":True})
    dm=float(res.tvalues[0])
    fac=math.sqrt(max((n+1-2*h+h*(h-1)/n)/n,EPSILON))
    dmh=dm*fac; p=float(2*(1-stats.t.cdf(abs(dmh),df=n-1)))
    return dm,dmh,p,n,bandwidth

pairs=[
    ("Regime-Calibrated TGARCH-X","Baseline TGARCH-X"),
    ("Regime-Aware Transformer","Baseline Transformer"),
    ("Regime-Aware Combined Forecast","Static Combined Forecast"),
    ("Baseline Transformer","EWMA"),("GRU","EWMA"),("PatchTST","EWMA"),
    ("Static Combined Forecast","GARCH")]
dmr=[]
for state in ["All","Low","High"]:
    x=daily_losses[(daily_losses["Weight_Scheme"]=="MarketCapitalisation")&
                   (daily_losses["State"]==state)]
    p=x.pivot(index="Date",columns="Model",values="QLIKE")
    for a,b in pairs:
        if a not in p or b not in p: continue
        cc=p[[a,b]].dropna(); dm,dh,pv,n,bw=dm_hac_hln(cc[a],cc[b])
        dmr.append({"State":state,"Model_1":a,"Model_2":b,
                    "Sign_Convention":"Negative favours Model_1; positive favours Model_2",
                    "Mean_Loss_Difference_M1_minus_M2":float((cc[a]-cc[b]).mean()),
                    "DM_HAC":dm,"DM_HLN":dh,"P_Value":pv,"N_Dates":n,"HAC_Bandwidth":bw})
dm=pd.DataFrame(dmr)
if not dm.empty:
    dm["Holm_P_Value"]=dm.groupby("State",group_keys=False)["P_Value"].transform(
        lambda x:multipletests(x.fillna(1).values,method="holm")[1])
save_csv(dm,"15_DM_HAC_HLN_Holm.csv")

# =============================================================================
# 17. MOVING-BLOCK BOOTSTRAP CONFIDENCE INTERVALS
# =============================================================================
def block_ci(a,b,reps=BOOTSTRAP_REPS,block=BLOCK_LENGTH,seed=2026):
    x=pd.concat([pd.Series(a).rename("a"),pd.Series(b).rename("b")],axis=1).dropna()
    d=(x["a"]-x["b"]).values; n=len(d)
    if n<20: return np.nan,np.nan,np.nan,n
    rng=np.random.default_rng(seed); vals=np.empty(reps)
    for i in range(reps):
        idx=moving_block_indices(n,min(block,n),rng); vals[i]=d[idx].mean()
    return float(d.mean()),float(np.quantile(vals,.025)),float(np.quantile(vals,.975)),n

cir=[]
for state in ["All","Low","High"]:
    x=daily_losses[(daily_losses["Weight_Scheme"]=="MarketCapitalisation")&
                   (daily_losses["State"]==state)]
    p=x.pivot(index="Date",columns="Model",values="QLIKE")
    for a,b in pairs[:3]:
        if a not in p or b not in p: continue
        cc=p[[a,b]].dropna()
        md,lo,hi,n=block_ci(cc[a].reset_index(drop=True),cc[b].reset_index(drop=True))
        cir.append({"State":state,"Model_1":a,"Model_2":b,
                    "Mean_QLIKE_Difference":md,"CI_2.5pct":lo,"CI_97.5pct":hi,
                    "N_Dates":n,"Bootstrap_Replications":BOOTSTRAP_REPS,"Block_Length":BLOCK_LENGTH})
save_csv(pd.DataFrame(cir),"16_BlockBootstrap_QLIKE_Difference_CI.csv")

# =============================================================================
# 18. MODEL CONFIDENCE SET AND SUPERIOR PREDICTIVE ABILITY — 5,000 REPS
# =============================================================================
mcs_meta=[]; mcs_rows=[]; spa_rows=[]
primary_names=list(PRIMARY_MODEL_COLS.keys())
for state in ["All","Low","High"]:
    x=daily_losses[(daily_losses["Weight_Scheme"]=="MarketCapitalisation")&
                   (daily_losses["State"]==state)&daily_losses["Model"].isin(primary_names)]
    p=x.pivot(index="Date",columns="Model",values="QLIKE").dropna()
    meta={"State":state,"N_Dates":len(p),"Replications":MCS_SPA_REPS}
    if len(p)<30 or not ARCH_BOOTSTRAP_AVAILABLE:
        meta["Status"]="Not run: insufficient dates or arch.bootstrap unavailable"
        mcs_meta.append(meta); continue
    try:
        obj=MCS(p,size=MCS_SIZE,reps=MCS_SPA_REPS,block_size=min(BLOCK_LENGTH,max(2,len(p)//5)),
                bootstrap="stationary",seed=20260807)
        obj.compute(); inc=set(obj.included); exc=set(obj.excluded)
        for m in p.columns:
            mcs_rows.append({"State":state,"Model":m,"Included_90pct_MCS":m in inc,
                             "Excluded":m in exc,"N_Dates":len(p),"Replications":MCS_SPA_REPS})
        meta["MCS_Status"]="Completed"
    except Exception as e:
        meta["MCS_Status"]="Failed"; meta["MCS_Error"]=repr(e)
    try:
        mean=p.mean(); bench=mean.idxmin(); alts=[c for c in p.columns if c!=bench]
        if alts:
            s=SPA(p[bench].values,p[alts].values,reps=MCS_SPA_REPS,
                  block_size=min(BLOCK_LENGTH,max(2,len(p)//5)),
                  studentize=True,bootstrap="stationary",seed=20260807)
            s.compute()
            pv=s.pvalues
            if hasattr(pv,"to_dict"): pvo={str(k):float(v) for k,v in pv.to_dict().items()}
            else: pvo={str(i):float(v) for i,v in enumerate(np.asarray(pv).reshape(-1))}
            spa_rows.append({"State":state,"Benchmark":bench,"Alternatives":"; ".join(alts),
                             "N_Dates":len(p),"Replications":MCS_SPA_REPS,
                             "SPA_PValues":json.dumps(pvo,sort_keys=True)})
        meta["SPA_Status"]="Completed"
    except Exception as e:
        meta["SPA_Status"]="Failed"; meta["SPA_Error"]=repr(e)
    mcs_meta.append(meta)
save_csv(pd.DataFrame(mcs_meta),"17_MCS_SPA_Metadata.csv")
save_csv(pd.DataFrame(mcs_rows),"17A_Model_Confidence_Set.csv")
save_csv(pd.DataFrame(spa_rows),"17B_Superior_Predictive_Ability.csv")

# =============================================================================
# 19. GIACOMINI-WHITE / GIACOMINI-ROSSI
# =============================================================================
def gw_test(d,highprob=None):
    z=pd.DataFrame({"d":pd.Series(d).astype(float).reset_index(drop=True)})
    z["lag_d"]=z["d"].shift(1)
    if highprob is not None: z["high_prob"]=pd.Series(highprob).astype(float).reset_index(drop=True)
    z=z.dropna()
    if len(z)<30: return np.nan,np.nan,len(z)
    cols=["lag_d"]+(["high_prob"] if "high_prob" in z else [])
    X=sm.add_constant(z[cols])
    r=sm.OLS(z["d"],X).fit(cov_type="HAC",cov_kwds={"maxlags":BLOCK_LENGTH,"use_correction":True})
    b=np.asarray(r.params); V=np.asarray(r.cov_params())
    stat=float(b.T@np.linalg.pinv(V)@b); pv=float(1-chi2.cdf(stat,len(b)))
    return stat,pv,len(z)

def gr_test(d,window_fraction=.30):
    d=pd.Series(d).dropna().astype(float).reset_index(drop=True); n=len(d)
    if n<40: return np.nan,2.770,3.030,n
    w=max(20,int(n*window_fraction)); vals=[]
    for st in range(n-w+1):
        a=d.iloc[st:st+w].values
        r=sm.OLS(a,np.ones((len(a),1))).fit(cov_type="HAC",
             cov_kwds={"maxlags":min(BLOCK_LENGTH,max(1,len(a)//5))})
        vals.append(float(r.tvalues[0]))
    return float(np.max(np.abs(vals))),2.770,3.030,n

dlall=daily_losses[(daily_losses["Weight_Scheme"]=="MarketCapitalisation")&
                   (daily_losses["State"]=="All")]
piv=dlall.pivot(index="Date",columns="Model",values="QLIKE")
mhp=(panel_eval[panel_eval["Evaluation_Split"]=="FinalTest"].groupby("Date")
     .apply(lambda g:weighted_mean_safe(g["Forecast_Probability_High"],g["Market_Weight_Daily"]))
     .rename("HighProb"))
gwgr=[]
for a,b in pairs[:3]:
    if a not in piv or b not in piv: continue
    cc=piv[[a,b]].dropna(); d=cc[a]-cc[b]; hp=mhp.reindex(cc.index)
    gs,gp,n=gw_test(d.reset_index(drop=True),hp.reset_index(drop=True))
    gm,c10,c5,nn=gr_test(d)
    gwgr.append({"Model_1":a,"Model_2":b,"GW_Statistic":gs,"GW_P_Value":gp,
                 "GR_MaxAbs":gm,"GR_Critical_10pct":c10,"GR_Critical_5pct":c5,
                 "GR_Unstable_10pct":bool(np.isfinite(gm) and gm>c10),
                 "GR_Unstable_5pct":bool(np.isfinite(gm) and gm>c5),"N_Dates":n})
save_csv(pd.DataFrame(gwgr),"18_GiacominiWhite_GiacominiRossi.csv")

# =============================================================================
# 20. MINCER-ZARNOWITZ 1-DAY AND 5-DAY
# =============================================================================
def mz(y,f):
    d=pd.DataFrame({"y":y,"f":f}).dropna()
    if len(d)<20: return {}
    r=sm.OLS(d["y"],sm.add_constant(d["f"])).fit(cov_type="HAC",
          cov_kwds={"maxlags":BLOCK_LENGTH,"use_correction":True})
    b=r.params.values; V=r.cov_params().values
    diff=b-np.array([0.,1.]); stat=float(diff.T@np.linalg.pinv(V)@diff)
    return {"N":len(d),"Alpha":float(b[0]),"Beta":float(b[1]),"R2":float(r.rsquared),
            "Joint_Alpha0_Beta1_Chi2":stat,"Joint_P_Value":float(1-chi2.cdf(stat,2))}
mzrec=[]; finalp=panel_eval[panel_eval["Evaluation_Split"]=="FinalTest"].copy()
for m,c in PRIMARY_MODEL_COLS.items():
    o=mz(finalp["Actual_Variance"],finalp[c])
    if o: mzrec.append({"Horizon":"1-day","Model":m,**o})
    aa=[]
    for _,g in finalp[["Sector","Date","Actual_Variance",c]].dropna().groupby("Sector"):
        g=g.sort_values("Date").copy()
        g["y5"]=g["Actual_Variance"].rolling(5,min_periods=5).sum()
        g["f5"]=g[c].rolling(5,min_periods=5).sum(); aa.append(g[["y5","f5"]])
    if aa:
        z=pd.concat(aa,ignore_index=True).dropna(); o=mz(z["y5"],z["f5"])
        if o: mzrec.append({"Horizon":"5-day","Model":m,**o})
save_csv(pd.DataFrame(mzrec),"19_Mincer_Zarnowitz_1Day_5Day.csv")

# =============================================================================
# 21. MODERATION: CONTINUOUS HIGH-REGIME PROBABILITY + ROBUST INFERENCE
# =============================================================================
print("\n"+"="*100)
print("MODERATION: MODEL-SPECIFIC LOSS SENSITIVITY TO HIGH-REGIME PROBABILITY")
print("="*100)

modnames=list(PRIMARY_MODEL_COLS.keys())
mod=loss_long[
    (loss_long["Evaluation_Split"]=="FinalTest")&
    loss_long["Model"].isin(modnames)
].copy()
mod["HighProb"]=pd.to_numeric(
    mod["Forecast_Probability_High"],errors="coerce").clip(0,1)
mod["HighHard"]=pd.to_numeric(mod["Forecast_Regime"],errors="coerce")
mod["LogQLIKE"]=np.log1p(mod["QLIKE"].clip(lower=0))
mod["Date_Str"]=mod["Date"].dt.strftime("%Y-%m-%d")
mod["Model"]=pd.Categorical(mod["Model"],categories=[
    "Baseline TGARCH-X","Static Combined Forecast","Regime-Aware Combined Forecast",
    "Regime-Calibrated TGARCH-X","Baseline Transformer","Regime-Aware Transformer"])
mod["Sector"]=pd.Categorical(mod["Sector"])

# Primary moderation specification uses the forecast-time filtered probability
# rather than a hard state indicator.
formula=(
    "LogQLIKE ~ C(Model, Treatment(reference='Baseline TGARCH-X')) * HighProb "
    "+ Actual_Variance + Volume_Return_Lag1 + C(Sector)"
)
mr=mod.dropna(subset=[
    "LogQLIKE","HighProb","Actual_Variance","Volume_Return_Lag1"
]).copy()

ols=smf.ols(formula,data=mr).fit()
sc=pd.Categorical(mr["Sector"]).codes
dc=pd.Categorical(mr["Date"]).codes
covtw,_,_=cov_cluster_2groups(ols,sc,dc)
setw=np.sqrt(np.diag(covtw))
beta=np.asarray(ols.params)
tt=beta/setw
dfcluster=max(min(mr["Sector"].nunique()-1,mr["Date"].nunique()-1),1)
ptw=2*(1-stats.t.cdf(np.abs(tt),df=dfcluster))

mres=pd.DataFrame({
    "Term":ols.params.index,"Estimate":beta,
    "TwoWayCluster_SE":setw,"TwoWayCluster_t":tt,"TwoWayCluster_P":ptw
})
mres["CI_Lower_95"]=mres["Estimate"]-1.96*mres["TwoWayCluster_SE"]
mres["CI_Upper_95"]=mres["Estimate"]+1.96*mres["TwoWayCluster_SE"]
mres["Regime_Variable"]="Forecast-time filtered P(High)"
save_csv(mres,"20_Moderation_Probability_TwoWayCluster.csv")

save_csv(pd.DataFrame([{
    "N":int(ols.nobs),"R2":ols.rsquared,"Adjusted_R2":ols.rsquared_adj,
    "AIC":ols.aic,"BIC":ols.bic,
    "Sector_Clusters":mr["Sector"].nunique(),
    "Date_Clusters":mr["Date"].nunique(),
    "Regime_Variable":"Forecast-time filtered P(High)",
    "Inference":"Two-way sector/date clustered covariance"
}]),"20A_Moderation_Probability_Fit_Statistics.csv")

def webb_weights(rng,n):
    support=np.array([
        -np.sqrt(1.5),-1.,-np.sqrt(.5),np.sqrt(.5),1.,np.sqrt(1.5)])
    return rng.choice(support,size=n,replace=True)

def wild_cluster_coef_p(X,y,bhat,j,clusters,reps=WILD_CLUSTER_REPS,seed=20260808):
    X=np.asarray(X,float); y=np.asarray(y,float); bhat=np.asarray(bhat,float)
    inv=np.linalg.pinv(X.T@X)
    R=np.zeros((1,X.shape[1])); R[0,j]=1.
    corr=inv@R.T@np.linalg.pinv(R@inv@R.T)@(R@bhat)
    br=bhat-corr.reshape(-1)
    fit=X@br; er=y-fit
    uc=np.unique(clusters)
    rng=np.random.default_rng(seed+j)
    vals=np.empty(reps)
    for k in range(reps):
        wd=webb_weights(rng,len(uc))
        mp=dict(zip(uc,wd))
        yy=fit+er*np.array([mp[c] for c in clusters])
        vals[k]=(inv@X.T@yy)[j]
    return float((1+np.sum(np.abs(vals)>=abs(bhat[j])))/(reps+1))

wild=[]
if RUN_WILD_CLUSTER_BOOTSTRAP:
    X=np.asarray(ols.model.exog,float)
    y=np.asarray(ols.model.endog,float)
    for j,name in enumerate(ols.params.index):
        if ":HighProb" not in name:
            continue
        wild.append({
            "Term":name,"Estimate":beta[j],
            "WildCluster_Webb_P":wild_cluster_coef_p(X,y,beta,j,sc),
            "Clusters":mr["Sector"].nunique(),
            "Replications":WILD_CLUSTER_REPS,
            "Null":"Probability interaction coefficient = 0"
        })
save_csv(pd.DataFrame(wild),"20B_Moderation_Probability_WildCluster_Webb.csv")

dfe=pd.DataFrame()
if RUN_DATE_FE_SENSITIVITY:
    try:
        rr=smf.ols(formula+" + C(Date_Str)",data=mr).fit(
            cov_type="cluster",
            cov_kwds={"groups":sc,"use_correction":True})
        dd=pd.DataFrame({
            "Term":rr.params.index,"Estimate":rr.params.values,
            "SectorCluster_SE":rr.bse.values,
            "SectorCluster_P":rr.pvalues.values
        })
        dfe=dd[dd["Term"].str.contains(":HighProb",regex=False)].copy()
        dfe["Sensitivity"]="Date fixed effects + sector-clustered SE"
    except Exception as ex:
        dfe=pd.DataFrame([{"Error":repr(ex)}])
save_csv(dfe,"20C_Moderation_Probability_DateFE_Sensitivity.csv")

# Hard-state interaction is retained as a sensitivity analysis only.
hard_formula=(
    "LogQLIKE ~ C(Model, Treatment(reference='Baseline TGARCH-X')) * HighHard "
    "+ Actual_Variance + Volume_Return_Lag1 + C(Sector)"
)
try:
    hr=mod.dropna(subset=[
        "LogQLIKE","HighHard","Actual_Variance","Volume_Return_Lag1"]).copy()
    hfit=smf.ols(hard_formula,data=hr).fit(
        cov_type="cluster",
        cov_kwds={"groups":pd.Categorical(hr["Sector"]).codes,
                  "use_correction":True})
    hres=pd.DataFrame({
        "Term":hfit.params.index,"Estimate":hfit.params.values,
        "SectorCluster_SE":hfit.bse.values,
        "SectorCluster_P":hfit.pvalues.values
    })
    hres=hres[hres["Term"].str.contains(":HighHard",regex=False)].copy()
    hres["Sensitivity"]="Hard 0/1 forecast regime; sector-clustered SE"
except Exception as ex:
    hres=pd.DataFrame([{"Error":repr(ex)}])
save_csv(hres,"20D_Moderation_HardState_Sensitivity.csv")

# =============================================================================
# 22. PARKINSON ALTERNATIVE VOLATILITY PROXY
# =============================================================================
park=build_loss_long(panel_eval,"Parkinson_Variance",
                     "Parkinson range-based evaluation target",
                     "alternative evaluation target; forecasts fixed")
save_csv(park,"21_Parkinson_Target_Losses.csv")
pa=[]
for ws in WEIGHT_SCHEMES:
    for st in [None,0,1]: pa.append(accuracy(park,ws,"FinalTest",st))
parkacc=pd.concat(pa,ignore_index=True)
parkacc["Robustness_Type"]="alternative evaluation target; forecasts fixed"
save_csv(parkacc,"21A_Parkinson_Target_Accuracy.csv")

# =============================================================================
# 23. THREE-STATE ZERO-NEUTRAL STUDENT-t HMM ROBUSTNESS
# =============================================================================
@njit
def _logsum3_numba(a,b,c):
    m=max(a,b,c)
    return m+math.log(math.exp(a-m)+math.exp(b-m)+math.exp(c-m))

@njit
def _hmm3_transition(theta):
    P=np.empty((3,3))
    for i in range(3):
        a=theta[2*i]; b=theta[2*i+1]
        m=max(a,b,0.0)
        ea=math.exp(a-m); eb=math.exp(b-m); ec=math.exp(-m)
        ss=ea+eb+ec
        P[i,0]=ea/ss; P[i,1]=eb/ss; P[i,2]=ec/ss
    return P

@njit
def _hmm3_stationary(P):
    pi=np.array([1/3,1/3,1/3],dtype=np.float64)
    for _ in range(250):
        nxt=np.empty(3)
        for j in range(3):
            nxt[j]=pi[0]*P[0,j]+pi[1]*P[1,j]+pi[2]*P[2,j]
        ss=nxt[0]+nxt[1]+nxt[2]
        pi=nxt/ss
    return pi

@njit
def _hmm3_student_nll(theta,z,zero):
    P=_hmm3_transition(theta)
    pi=_hmm3_stationary(P)

    s0=math.exp(theta[6])
    s1=s0+math.exp(theta[7])
    s2=s1+math.exp(theta[8])
    nu=2.05+math.exp(theta[9])
    scales=np.array([s0,s1,s2])

    alpha=np.empty(3)
    for j in range(3):
        e=0.0 if zero[0] else _student_logpdf_std(z[0],scales[j],nu)
        alpha[j]=math.log(max(pi[j],1e-14))+e
    ll=_logsum3_numba(alpha[0],alpha[1],alpha[2])
    for j in range(3): alpha[j]-=ll
    total=ll

    for t in range(1,len(z)):
        pred=np.empty(3)
        for j in range(3):
            pred[j]=_logsum3_numba(
                alpha[0]+math.log(max(P[0,j],1e-14)),
                alpha[1]+math.log(max(P[1,j],1e-14)),
                alpha[2]+math.log(max(P[2,j],1e-14)))
        for j in range(3):
            e=0.0 if zero[t] else _student_logpdf_std(z[t],scales[j],nu)
            alpha[j]=pred[j]+e
        lt=_logsum3_numba(alpha[0],alpha[1],alpha[2])
        for j in range(3): alpha[j]-=lt
        total+=lt

    return -total if math.isfinite(total) else 1e50

@njit
def _hmm3_student_filter(theta,z,zero):
    P=_hmm3_transition(theta)
    f=_hmm3_stationary(P)
    s0=math.exp(theta[6])
    s1=s0+math.exp(theta[7])
    s2=s1+math.exp(theta[8])
    nu=2.05+math.exp(theta[9])
    scales=np.array([s0,s1,s2])
    out=np.empty((len(z),3))

    for t in range(len(z)):
        if t>0:
            pred=np.empty(3)
            for j in range(3):
                pred[j]=f[0]*P[0,j]+f[1]*P[1,j]+f[2]*P[2,j]
            f=pred
        if not zero[t]:
            logs=np.empty(3)
            for j in range(3):
                logs[j]=_student_logpdf_std(z[t],scales[j],nu)
            mx=max(logs[0],logs[1],logs[2])
            for j in range(3):
                f[j]*=math.exp(logs[j]-mx)
        ss=f[0]+f[1]+f[2]
        if ss<=0.0 or not math.isfinite(ss):
            f=np.array([1/3,1/3,1/3],dtype=np.float64)
        else:
            f=f/ss
        out[t,0]=f[0]; out[t,1]=f[1]; out[t,2]=f[2]
    return out

# JIT warm-up.
_=_hmm3_student_nll(
    np.array([2.5,0.,0.,2.5,-2.5,-2.5,
              np.log(.25),np.log(.35),np.log(.55),np.log(5.95)]),
    _dummy_z,_dummy_zero)

def _fit_hmm3_sector(sd,fit_end):
    sd=sd.sort_values("Date").copy().reset_index(drop=True)
    sector=sd["Sector"].iloc[0]
    fit=(sd["Date"]<=fit_end).values
    y=pd.to_numeric(sd["Sector_Return"],errors="coerce").values.astype(float)
    yfit=y[fit]
    zero_fit=np.isclose(yfit,0.0,atol=HMM_ZERO_ATOL)
    nz=~zero_fit

    if int(fit.sum())<MIN_MARKOV_OBS or int(nz.sum())<MIN_MARKOV_OBS//2:
        return None,pd.DataFrame(),{
            "Sector":sector,"Accepted":False,
            "Reason":"Insufficient fitting/non-zero observations"
        }

    mu=float(np.mean(yfit[nz]))
    sig=float(np.std(yfit[nz],ddof=1))
    if not np.isfinite(sig) or sig<=EPSILON:
        return None,pd.DataFrame(),{
            "Sector":sector,"Accepted":False,"Reason":"Near-zero nonzero-return SD"
        }

    z=(y-mu)/sig
    zfit=z[fit]
    zero_all=np.isclose(y,0.0,atol=HMM_ZERO_ATOL)
    seed=sum(map(ord,str(sector)))+3301
    rng=np.random.default_rng(seed)
    attempts=[]; candidates=[]

    base_trans=np.array([2.5,0.0,0.0,2.5,-2.5,-2.5])
    for j in range(HMM_THREE_STATE_STARTS):
        trans=base_trans+rng.normal(0,0.55,6)
        s0=max(0.08,0.22*np.exp(rng.normal(0,.25)))
        s1=max(s0+.05,0.60*np.exp(rng.normal(0,.25)))
        s2=max(s1+.05,1.30*np.exp(rng.normal(0,.25)))
        nu0=float(np.clip(6*np.exp(rng.normal(0,.25)),2.6,30))
        x0=np.r_[trans,np.log(s0),np.log(max(s1-s0,.04)),
                 np.log(max(s2-s1,.04)),np.log(max(nu0-2.05,.05))]
        bounds=[(-8,8)]*6+[(-8,3),(-8,4),(-8,4),(-6,8)]
        try:
            res=minimize(
                lambda th:_hmm3_student_nll(
                    np.asarray(th,dtype=np.float64),zfit,zero_fit.astype(np.bool_)),
                x0,method="L-BFGS-B",bounds=bounds,
                options={"maxiter":HMM_MAXITER_3STATE,"ftol":1e-11,"gtol":1e-7})
            P=np.asarray(_hmm3_transition(np.asarray(res.x,dtype=np.float64)))
            probs=_hmm3_student_filter(
                np.asarray(res.x,dtype=np.float64),z,zero_all.astype(np.bool_))
            state=np.argmax(probs,axis=1)
            sf=state[fit]
            shares=[float(np.mean(sf==k)) for k in range(3)]
            av=pd.to_numeric(sd["Actual_Variance"],errors="coerce").values[fit]
            avmeans=[
                float(np.mean(av[sf==k])) if np.any(sf==k) else np.nan
                for k in range(3)]
            s0=float(np.exp(res.x[6]))
            s1=float(s0+np.exp(res.x[7]))
            s2=float(s1+np.exp(res.x[8]))
            nu=float(2.05+np.exp(res.x[9]))
            occupancy_ok=bool(min(shares)>=HMM_MIN_STATE_SHARE_3)
            trans_ok=bool(np.min(P)>=HMM_TRANSITION_EPS)
            scale_ok=bool(s1>s0 and s2>s1)
            accepted=bool(res.success and np.isfinite(res.fun)
                          and occupancy_ok and trans_ok and scale_ok)
            row={
                "Sector":sector,"Start":j+1,"Converged":bool(res.success),
                "Accepted":accepted,"NegLogLikelihood_StatePart":float(res.fun),
                "Low_State_Share":shares[0],"Medium_State_Share":shares[1],
                "High_State_Share":shares[2],
                "ActualVariance_Low":avmeans[0],
                "ActualVariance_Medium":avmeans[1],
                "ActualVariance_High":avmeans[2],
                "Sigma0":s0,"Sigma1":s1,"Sigma2":s2,"Nu":nu,
                "Min_Transition_Probability":float(np.min(P)),
                "Occupancy_OK":occupancy_ok,"Transition_OK":trans_ok,
                "Message":str(res.message)
            }
            attempts.append(row)
            if accepted:
                candidates.append((float(res.fun),res,probs,row,P))
        except Exception as ex:
            attempts.append({
                "Sector":sector,"Start":j+1,"Converged":False,
                "Accepted":False,"Error":repr(ex)
            })

    adf=pd.DataFrame(attempts)
    if not candidates:
        return None,adf,{
            "Sector":sector,"Accepted":False,
            "Reason":"No valid 3-state zero-neutral Student-t HMM solution"
        }

    _,res,probs,row,P=min(candidates,key=lambda x:x[0])
    state=np.argmax(probs,axis=1).astype(int)
    out=pd.DataFrame({
        "Date":sd["Date"],"Sector":sector,"State3":state,
        "Probability_Low":probs[:,0],
        "Probability_Medium":probs[:,1],
        "Probability_High":probs[:,2],
        "Method":"ZeroNeutral_StudentT_HMM3_FilteredForward",
        "Converged":True
    })
    out["Forecast_State3"]=out["State3"].shift(1)
    meta={
        "Sector":sector,"Accepted":True,
        "Fit_End":fit_end,"Fit_Observations":int(fit.sum()),
        "Zero_Return_Share":float(zero_fit.mean()),
        "Standardisation_Mean":mu,"Standardisation_SD":sig,
        "Selected_Start":int(row["Start"]),
        "Low_State_Share":row["Low_State_Share"],
        "Medium_State_Share":row["Medium_State_Share"],
        "High_State_Share":row["High_State_Share"],
        "Sigma0":row["Sigma0"],"Sigma1":row["Sigma1"],"Sigma2":row["Sigma2"],
        "Nu":row["Nu"],
        "Min_Transition_Probability":float(np.min(P))
    }
    return out,adf,meta

three=pd.DataFrame()
three_meta=[]; three_attempts=[]; failed3=[]
if RUN_3STATE_REGIME_ROBUSTNESS:
    ff=[]
    for sector in tqdm(sorted(features["Sector"].unique()),
                       desc="3-state zero-neutral Student-t HMM"):
        o,a,m=_fit_hmm3_sector(
            features[features["Sector"]==sector],BOUND["Validation_End"])
        three_attempts.append(a); three_meta.append(m)
        if o is None:
            failed3.append(sector)
        else:
            ff.append(o)
    if three_attempts:
        save_csv(pd.concat(three_attempts,ignore_index=True,sort=False),
                 "22B_ThreeState_ZeroNeutral_StudentT_Attempts.csv")
    save_csv(pd.DataFrame(three_meta),
             "22C_ThreeState_ZeroNeutral_StudentT_Meta.csv")
    if ff:
        three=pd.concat(ff,ignore_index=True)

save_csv(three,"22_Alternative_3State_ZeroNeutral_StudentT_Classification.csv")
save_csv(pd.DataFrame([{
    "Successful_Sectors":int(three["Sector"].nunique()) if not three.empty else 0,
    "Expected_Sectors":EXPECTED_SECTORS,
    "Failed_Sectors":"; ".join(failed3),
    "Interpretation":"Robustness specification; no fallback substituted for failed sectors."
}]),"22D_ThreeState_Robustness_Coverage.csv")

if not three.empty:
    tp=panel_eval.merge(
        three[["Date","Sector","Forecast_State3"]],
        on=["Date","Sector"],how="inner",validate="one_to_one")
    tr=[]
    for st in [0,1,2]:
        g=tp[
            (tp["Evaluation_Split"]=="FinalTest")&
            (tp["Forecast_State3"]==st)]
        for m,c in PRIMARY_MODEL_COLS.items():
            gg=g.dropna(subset=["Actual_Variance",c,"Market_Weight_Daily"])
            vals=[]
            for _,gd in gg.groupby("Date"):
                w=gd["Market_Weight_Daily"].values
                if np.nansum(w)>0:
                    vals.append(weighted_mean_safe(
                        qlike(gd["Actual_Variance"],gd[c]),w/np.nansum(w)))
            tr.append({
                "State3":st,"State_Label":["Low","Medium","High"][st],
                "Model":m,"QLIKE":np.nanmean(vals) if vals else np.nan,
                "N_Dates":gg["Date"].nunique(),
                "N_Sectors":gg["Sector"].nunique(),
                "Robustness_Type":"three-state zero-neutral Student-t HMM; forecasts fixed"
            })
    save_csv(pd.DataFrame(tr),"22A_ThreeState_Accuracy_Robustness.csv")

# =============================================================================
# 24. PRIMARY REGIME NO-FALLBACK / VALIDITY AUDIT
# =============================================================================
strict_regime_audit=pd.DataFrame(final_meta).copy()
strict_regime_audit["All_Primary_Sectors_Accepted"]=bool(
    len(strict_regime_audit)==EXPECTED_SECTORS
    and strict_regime_audit["Accepted"].fillna(False).all()
)
strict_regime_audit["Fallback_Used"]=False
strict_regime_audit["Model_Signature"]=HMM_MODEL_SIGNATURE
strict_regime_audit["ZeroReturn_Validity_All_Pass"]=bool(
    not decision_audit["Validity_Fail"].fillna(False).any()
)
strict_regime_audit["ProbabilityAware_Transformer_All_Sectors"]=bool(
    regtf["Sector"].nunique()==EXPECTED_SECTORS
)
save_csv(strict_regime_audit,"23_Primary_Regime_NoFallback_Validity_Audit.csv")

# =============================================================================
# 25. SEQUENCE-LENGTH SENSITIVITY: FULL TRANSFORMER RE-ESTIMATION
# =============================================================================
def _ensemble_from_detail_file(detail_df, seq_len):
    """
    Reconstruct the ensemble from v5.04 per-seed detail output. This lets the
    completed sequence-10 sensitivity run be reused even though v5.04 did not
    save its ensemble as a separate file.
    """
    if detail_df is None or detail_df.empty:
        return pd.DataFrame()
    if "Record_Type" not in detail_df.columns:
        return pd.DataFrame()
    ps = detail_df[detail_df["Record_Type"].eq("PerSeedForecast")].copy()
    if ps.empty:
        return pd.DataFrame()
    if "Date" in ps.columns:
        ps["Date"] = parse_dates(ps["Date"])
    ens = (
        ps.groupby(["Date","Sector","Model"],as_index=False)
        .agg(
            Forecast=("Forecast","mean"),
            Forecast_Seed_SD=("Forecast","std"),
            N_Seeds=("Seed","nunique"),
            Actual_Variance=("Actual_Variance","first"),
            Sequence_Length=("Sequence_Length","first"),
        )
    )
    if ens["Sector"].nunique() != EXPECTED_SECTORS:
        return pd.DataFrame()
    if pd.to_numeric(ens["N_Seeds"],errors="coerce").min() < len(NEURAL_SEEDS):
        return pd.DataFrame()
    return ens

seqrec=[]
if RUN_SEQUENCE_SENSITIVITY:
    for sl in SEQUENCE_SENSITIVITY:
        if sl==PRIMARY_SEQUENCE_LENGTH:
            sdf=neural["Transformer"].copy()
        else:
            # First try to reuse a fully completed v5.04 sensitivity run.
            existing_detail = _read_resume_csv(
                f"24B_Transformer_Details_Seq{sl}.csv"
            )
            sdf = _ensemble_from_detail_file(existing_detail, sl)

            if not sdf.empty:
                print(
                    f"RESUME: complete Transformer sequence-{sl} sensitivity "
                    "reconstructed from prior seed-level details."
                )
                stune = _read_resume_csv(
                    f"24A_Transformer_Tuning_Seq{sl}.csv",
                    parse_date_cols=(),
                )
                sdetail = existing_detail
            else:
                # Sector-level checkpointing means that any future disconnect
                # resumes at the next unfinished sector.
                sdf,stune,sdetail=run_neural(
                    "Transformer",
                    "Actual_Variance",
                    sl,
                    stage=f"sequence_sensitivity_{sl}",
                )

            if stune is not None and not stune.empty:
                save_csv(stune,f"24A_Transformer_Tuning_Seq{sl}.csv")
            if sdetail is not None and not sdetail.empty:
                save_csv(sdetail,f"24B_Transformer_Details_Seq{sl}.csv")

        if sdf.empty:
            continue

        ee=panel_eval[
            ["Date","Sector","Evaluation_Split","Actual_Variance","Market_Weight_Daily"]
        ].merge(
            sdf[["Date","Sector","Forecast"]],
            on=["Date","Sector"],
            how="inner",
        )
        ee=ee[ee["Evaluation_Split"]=="FinalTest"]
        vals=[]
        for _,gd in ee.groupby("Date"):
            w=gd["Market_Weight_Daily"].values
            if np.nansum(w)>0:
                vals.append(
                    weighted_mean_safe(
                        qlike(gd["Actual_Variance"],gd["Forecast"]),
                        w/np.nansum(w),
                    )
                )
        seqrec.append({
            "Sequence_Length":sl,
            "Model":"Baseline Transformer",
            "QLIKE":np.nanmean(vals) if vals else np.nan,
            "N_Dates":ee["Date"].nunique(),
            "N_SectorDays":len(ee),
            "Robustness_Type":"full neural re-estimation",
            "Neural_Seeds":";".join(map(str,NEURAL_SEEDS)),
        })

save_csv(pd.DataFrame(seqrec),"24_Sequence_Length_Sensitivity.csv")

# =============================================================================
# 26. 55/65/75 SPLIT RECLASSIFICATION SENSITIVITY
# =============================================================================
ss=[]; du=np.array(sorted(panel_eval["Date"].unique()))
for ratio in [.55,.65,.75]:
    ii=int(np.floor(len(du)*ratio)); ii=min(max(ii,20),len(du)-20)
    start=pd.Timestamp(du[ii]); g=panel_eval[panel_eval["Date"]>=start]
    for m,c in PRIMARY_MODEL_COLS.items():
        gg=g.dropna(subset=["Actual_Variance",c,"Market_Weight_Daily"]); vals=[]
        for _,gd in gg.groupby("Date"):
            w=gd["Market_Weight_Daily"].values
            if np.nansum(w)>0:
                vals.append(weighted_mean_safe(qlike(gd["Actual_Variance"],gd[c]),w/np.nansum(w)))
        ss.append({"Split_Ratio":ratio,"Reclassified_FinalTest_Start":start,"Model":m,
                   "QLIKE":np.nanmean(vals) if vals else np.nan,"N_Dates":gg["Date"].nunique(),
                   "Robustness_Type":"observation reclassification only; forecasts/weights fixed"})
save_csv(pd.DataFrame(ss),"25_Split_Point_Reclassification_Sensitivity.csv")

# =============================================================================
# 27. HIGH-VOLATILITY DETERIORATION DIAGNOSTICS
# =============================================================================
hd=[]
h=panel_eval[(panel_eval["Evaluation_Split"]=="FinalTest")&(panel_eval["Forecast_Regime"]==1)]
for sector,g in h.groupby("Sector"):
    g=g.dropna(subset=list(PRIMARY_MODEL_COLS.values()))
    if len(g)<5: continue
    etg=g["Actual_Variance"]-g["FC_TGARCH_X"]; etf=g["Actual_Variance"]-g["FC_Transformer"]
    hd.append({"Sector":sector,"N_High_FinalTest":len(g),
               "TGARCH_X_QLIKE":np.mean(qlike(g["Actual_Variance"],g["FC_TGARCH_X"])),
               "Transformer_QLIKE":np.mean(qlike(g["Actual_Variance"],g["FC_Transformer"])),
               "Static_Combined_QLIKE":np.mean(qlike(g["Actual_Variance"],g["FC_Combined_Static"])),
               "RegCal_TGARCH_X_QLIKE":np.mean(qlike(g["Actual_Variance"],g["FC_TGARCH_X_RegCal"])),
               "Regime_Transformer_QLIKE":np.mean(qlike(g["Actual_Variance"],g["FC_Transformer_Regime"])),
               "Regime_Combined_QLIKE":np.mean(qlike(g["Actual_Variance"],g["FC_Combined_Regime"])),
               "Component_Forecast_Correlation":g["FC_TGARCH_X"].corr(g["FC_Transformer"]),
               "Component_Error_Correlation":pd.Series(etg).corr(pd.Series(etf)),
               "Mean_Static_Transformer_Weight":g["Weight_Transformer_Static"].mean(),
               "Mean_Regime_Transformer_Weight":g["Weight_Transformer_Regime"].mean(),
               "Mean_TGARCH_X_Calibration_Factor":g["TGARCH_X_Calibration_Factor"].mean(),
               "Mean_Baseline_Transformer_Seed_SD":g["Transformer_Seed_SD"].mean() if "Transformer_Seed_SD" in g else np.nan,
               "Mean_Regime_Transformer_Seed_SD":g["Regime_Transformer_Seed_SD"].mean() if "Regime_Transformer_Seed_SD" in g else np.nan})
save_csv(pd.DataFrame(hd),"26_HighVolatility_Deterioration_Diagnostics.csv")

# =============================================================================
# 28. ECONOMIC SIGNIFICANCE: ILLUSTRATIVE VOLATILITY TARGETING
# =============================================================================
eco=[]
if RUN_ECONOMIC_SIGNIFICANCE:
    targ=.10/np.sqrt(252)
    for m,c in PRIMARY_MODEL_COLS.items():
        g=finalp.dropna(subset=["Sector_Return",c,"Market_Weight_Daily"]).copy()
        g["Position"]=(targ/np.sqrt(g[c].clip(lower=EPSILON))).clip(0,2)
        g["Strategy_Return"]=g["Position"]*g["Sector_Return"]; dr=[]
        for date,gd in g.groupby("Date"):
            w=gd["Market_Weight_Daily"].values
            if np.nansum(w)>0:
                wn=w/np.nansum(w)
                dr.append({"Date":date,"Return":weighted_mean_safe(gd["Strategy_Return"],wn),
                           "Position":weighted_mean_safe(gd["Position"],wn)})
        d=pd.DataFrame(dr).dropna()
        if len(d)<20: continue
        am=d["Return"].mean()*252; av=d["Return"].std(ddof=1)*np.sqrt(252)
        eco.append({"Model":m,"N_Dates":len(d),"Target_Annualised_Volatility":.10,
                    "Realised_Annualised_Volatility":av,"Annualised_Mean_Return":am,
                    "Illustrative_Sharpe_NoRiskFree":am/av if av>0 else np.nan,
                    "Mean_Exposure":d["Position"].mean(),"Max_Exposure_Cap":2.,
                    "Interpretation":"Illustrative volatility-targeting diagnostic; no transaction costs."})
save_csv(pd.DataFrame(eco),"27_Volatility_Targeting_Diagnostic.csv")

# =============================================================================
# 29. ROLLING LOSS DIFFERENTIALS AND FIGURES
# =============================================================================
rolls=[]
p=dlall.pivot(index="Date",columns="Model",values="QLIKE").sort_index()
for a,b in pairs[:3]:
    if a not in p or b not in p: continue
    d=(p[a]-p[b]).dropna(); r=d.rolling(30,min_periods=15)
    z=pd.DataFrame({"Date":d.index,"Pair":f"{a} minus {b}","Loss_Difference":d.values,
                    "Rolling30_Mean":r.mean().values,
                    "Rolling30_SE":(r.std(ddof=1)/np.sqrt(r.count())).values})
    z["CI_Lower"]=z["Rolling30_Mean"]-1.96*z["Rolling30_SE"]
    z["CI_Upper"]=z["Rolling30_Mean"]+1.96*z["Rolling30_SE"]; rolls.append(z)
roll=pd.concat(rolls,ignore_index=True) if rolls else pd.DataFrame()
save_csv(roll,"28_Rolling_QLIKE_Differentials.csv")
if not roll.empty:
    for pair,g in roll.groupby("Pair"):
        fig,ax=plt.subplots(figsize=(10,5)); ax.plot(g["Date"],g["Rolling30_Mean"])
        ax.fill_between(g["Date"],g["CI_Lower"],g["CI_Upper"],alpha=.2); ax.axhline(0,lw=1)
        ax.set_title(pair); ax.set_ylabel("QLIKE difference"); ax.set_xlabel("Date"); fig.tight_layout()
        safe=re.sub(r"[^A-Za-z0-9]+","_",pair)[:100]
        fig.savefig(PLOT_DIR/f"Rolling_QLIKE_{safe}.png",dpi=300)
        fig.savefig(PLOT_DIR/f"Rolling_QLIKE_{safe}.pdf"); plt.close(fig)

try:
    heat=sector_acc.pivot(index="Sector",columns="Model",values="QLIKE")
    fig,ax=plt.subplots(figsize=(14,7)); im=ax.imshow(np.log1p(heat.values),aspect="auto")
    ax.set_xticks(np.arange(len(heat.columns))); ax.set_xticklabels(heat.columns,rotation=75,ha="right")
    ax.set_yticks(np.arange(len(heat.index))); ax.set_yticklabels(heat.index)
    ax.set_title("Sector-level log(1 + QLIKE)"); fig.colorbar(im,ax=ax,label="log(1 + QLIKE)")
    fig.tight_layout(); fig.savefig(PLOT_DIR/"Sector_Level_QLIKE_Heatmap.png",dpi=300)
    fig.savefig(PLOT_DIR/"Sector_Level_QLIKE_Heatmap.pdf"); plt.close(fig)
except Exception as e:
    print("Heatmap warning:",e)

# =============================================================================
# 30. REPRODUCIBILITY, SAMPLE FLOW AND REVIEWER-COVERAGE MATRIX
# =============================================================================
conf={
    "CODE_VERSION":CODE_VERSION,"R_CODE_VERSION":r_version,
    "SAMPLE_START":str(SAMPLE_START.date()),"SAMPLE_END":str(SAMPLE_END.date()),
    "EXPECTED_FIRMS":EXPECTED_FIRMS,"EXPECTED_SECTORS":EXPECTED_SECTORS,
    "PRIMARY_SEQUENCE_LENGTH":PRIMARY_SEQUENCE_LENGTH,"NEURAL_SEEDS":NEURAL_SEEDS,
    "TUNING_SEED":TUNING_SEED,"MAX_EPOCHS":MAX_EPOCHS,
    "EARLY_STOPPING_PATIENCE":EARLY_STOPPING_PATIENCE,"BATCH_SIZE":BATCH_SIZE,
    "FINALTEST_START_RATIO":FINALTEST_START_RATIO,
    "STACKTRAIN_SHARE_OF_PRETEST":STACKTRAIN_SHARE_OF_PRETEST,
    "VALIDATION_SHARE_OF_FITPOOL":VALIDATION_SHARE_OF_FITPOOL,
    "BOOTSTRAP_REPS":BOOTSTRAP_REPS,"MCS_SPA_REPS":MCS_SPA_REPS,
    "WILD_CLUSTER_REPS":WILD_CLUSTER_REPS,"BLOCK_LENGTH":BLOCK_LENGTH,
    "QUICK_TEST":QUICK_TEST,"RESUME_COMPLETED_OUTPUTS":RESUME_COMPLETED_OUTPUTS,"PERSIST_CHECKPOINTS_TO_DRIVE":PERSIST_CHECKPOINTS_TO_DRIVE,"SINGLE_SESSION_FROM_SCRATCH":True,"INCLUDE_EXACT_INPUTS_IN_REPRO_BUNDLE":INCLUDE_EXACT_INPUTS_IN_REPRO_BUNDLE,"REQUIRE_GPU_FOR_FULL_COLAB_RUN":REQUIRE_GPU_FOR_FULL_COLAB_RUN,"HMM_MODEL_SIGNATURE":HMM_MODEL_SIGNATURE,"HMM_MIN_STATE_SHARE":HMM_MIN_STATE_SHARE,"HMM_MIN_STATE_SHARE_3":HMM_MIN_STATE_SHARE_3,"HMM_RARE_STATE_WARNING_SHARE":HMM_RARE_STATE_WARNING_SHARE,"HMM_MIN_VARIANCE_RATIO":HMM_MIN_VARIANCE_RATIO,"HMM_ZERO_ATOL":HMM_ZERO_ATOL,"HMM_TWO_STATE_STARTS":HMM_TWO_STATE_STARTS,"HMM_THREE_STATE_STARTS":HMM_THREE_STATE_STARTS,"RUN_ZERO_NEUTRAL_GAUSSIAN_ROBUSTNESS":RUN_ZERO_NEUTRAL_GAUSSIAN_ROBUSTNESS,"RUN_CONVENTIONAL_STUDENTT_DIAGNOSTIC":RUN_CONVENTIONAL_STUDENTT_DIAGNOSTIC,"RUN_ZERO_RETURN_REGIME_PREFLIGHT":RUN_ZERO_RETURN_REGIME_PREFLIGHT,"STOP_ON_ZERO_RETURN_REGIME_FAILURE":STOP_ON_ZERO_RETURN_REGIME_FAILURE,"ZERO_LOW_DOMINANCE_THRESHOLD":ZERO_LOW_DOMINANCE_THRESHOLD,"LOW_ZERO_CAPTURE_THRESHOLD":LOW_ZERO_CAPTURE_THRESHOLD,"ZERO_REGIME_PHI_THRESHOLD":ZERO_REGIME_PHI_THRESHOLD,"ZERO_REGIME_ODDS_RATIO_THRESHOLD":ZERO_REGIME_ODDS_RATIO_THRESHOLD,"TensorFlow":tf.__version__,"Python":sys.version,
    "Platform":platform.platform()}
with open(OUTPUT_DIR/"29_Run_Configuration.json","w",encoding="utf-8") as f:
    json.dump(conf,f,indent=2,default=str)

complete=panel_eval[list(PRIMARY_MODEL_COLS.values())].notna().all(axis=1)
flow=pd.DataFrame([
    {"Stage":"R authoritative target","Sector_Days":len(target),"Dates":target["Date"].nunique(),"Sectors":target["Sector"].nunique()},
    {"Stage":"R econometric support","Sector_Days":len(econ),"Dates":econ["Date"].nunique(),"Sectors":econ["Sector"].nunique()},
    {"Stage":"StackTrain evaluation panel","Sector_Days":int((panel_eval["Evaluation_Split"]=="StackTrain").sum()),
     "Dates":panel_eval.loc[panel_eval["Evaluation_Split"]=="StackTrain","Date"].nunique(),
     "Sectors":panel_eval.loc[panel_eval["Evaluation_Split"]=="StackTrain","Sector"].nunique()},
    {"Stage":"FinalTest evaluation panel","Sector_Days":int((panel_eval["Evaluation_Split"]=="FinalTest").sum()),
     "Dates":panel_eval.loc[panel_eval["Evaluation_Split"]=="FinalTest","Date"].nunique(),
     "Sectors":panel_eval.loc[panel_eval["Evaluation_Split"]=="FinalTest","Sector"].nunique()},
    {"Stage":"FinalTest primary-six complete cases",
     "Sector_Days":int(((panel_eval["Evaluation_Split"]=="FinalTest")&complete).sum()),
     "Dates":panel_eval.loc[(panel_eval["Evaluation_Split"]=="FinalTest")&complete,"Date"].nunique(),
     "Sectors":panel_eval.loc[(panel_eval["Evaluation_Split"]=="FinalTest")&complete,"Sector"].nunique()},
])
save_csv(flow,"30_Observation_Flow.csv")

coverage=pd.DataFrame([
["Authoritative common target","Imports R 20_Common_Target_for_Python.csv; no Python reconstruction","Complete"],
["Econometric benchmarks","Historical variance, EWMA, GARCH, GJR-GARCH, TGARCH, TGARCH-X","Complete"],
["Established recurrent benchmark","GRU","Complete"],
["Recent time-series benchmark","PatchTST","Complete"],
["Hyperparameter transparency","Chronological tuning grids and selected epochs/parameter counts","Complete"],
["Random-seed stability","Five fixed seeds, per-seed metrics and forecast SD","Complete"],
["Training-only scaling","Train scaler for tuning; Train+Validation scaler for final fit","Complete"],
["Common chronological split","Global Train/Validation/StackTrain/FinalTest dates","Complete"],
["Regime leakage control","Tuning HMM fitted through Train_End; final HMM refitted through Validation_End; both filtered forward; forecast-time probability is lagged","Complete"],
["Correct TGARCH terminology","Regime-Calibrated TGARCH-X","Complete"],
["Soft regime information","Filtered P(High) enters the Transformer as a lagged sequence feature and conditions TGARCH-X calibration/combination weights","Complete"],
["Combination robustness","Static convex/equal/unconstrained benchmarks plus nested-constant/equal probability-component combinations","Complete"],
["Sector-level evidence","Sector QLIKE/RMSE/MAE/ranks and heatmap","Complete"],
["Weight robustness","Market-cap, equal-sector, capped market-cap","Complete"],
["Alternative proxy","Parkinson range-based target evaluation","Complete"],
["DM inference","HAC + HLN + Holm","Complete"],
["MCS/SPA","Stationary bootstrap, 5,000 reps","Complete"],
["Confidence intervals","Moving-block bootstrap intervals","Complete"],
["Moderation dependence","Continuous filtered high-regime probability interactions with two-way sector/date clustered covariance","Complete"],
["Few sector clusters","Webb-weight wild-cluster bootstrap on probability-interaction terms","Complete"],
["Date-FE sensitivity","Date fixed effects + sector-clustered SE","Complete"],
["Alternative regimes","Zero-neutral Gaussian HMM, conventional Student-t diagnostic, and three-state zero-neutral Student-t HMM robustness","Complete"],
["Primary regime estimation","Two-state state-independent zero-hurdle Student-t HMM with deterministic multi-start optimisation and no fallback","Complete"],
["Regime validity","Zero-return/state contingency, Fisher test, phi coefficient, state persistence, variance separation, rare-state warning, and conservative preflight stop","Complete"],
["Sequence sensitivity","10/20/40, full Transformer re-estimation","Complete"],
["Split sensitivity","55/65/75 reclassification, explicitly labelled as fixed forecasts/weights","Complete"],
["High-volatility diagnosis","Component errors/correlations/weights/calibration/seed dispersion","Complete"],
["Economic significance","Illustrative volatility-targeting diagnostic","Complete"],
["Rolling instability","Rolling QLIKE differentials and confidence bands","Complete"],
["R rolling-window sensitivity","Separate R WINDOW profile still required","External R stage"],
["R distribution sensitivity","Separate R DISTRIBUTION profile still required","External R stage"],
["R volume-X sensitivity","Separate R X profile still required","External R stage"],
["Single-session reproducibility","Fresh inputs; no Google Drive; no prior-output reuse; exact inputs/environment/output manifest bundled together","Complete"],
["Manuscript/ethics/presentation revisions","Not a Python task","Manuscript stage"],
],columns=["Reviewer_Item","Implementation","Status"])
save_csv(coverage,"31_Reviewer_Recommendation_Coverage.csv")

hashrows=[]
if MASTER_FILE.exists(): hashrows.append({"File":str(MASTER_FILE),"SHA256":sha256_file(MASTER_FILE)})
if R_INPUT.exists() and R_INPUT.is_file(): hashrows.append({"File":str(R_INPUT),"SHA256":sha256_file(R_INPUT)})
save_csv(pd.DataFrame(hashrows),"32_Input_SHA256.csv")

manifest=[]
for pth in sorted(OUTPUT_DIR.rglob("*")):
    if pth.is_file() and pth.name!="33_Output_Manifest.csv":
        manifest.append({"Relative_Path":str(pth.relative_to(OUTPUT_DIR)),
                         "Bytes":pth.stat().st_size,"SHA256":sha256_file(pth)})
save_csv(pd.DataFrame(manifest),"33_Output_Manifest.csv")

print("\n"+"="*100); print("PIPELINE COMPLETE"); print("="*100)
print("Output:",OUTPUT_DIR.resolve())
print("R version verified:",r_version)
print("Python version:",CODE_VERSION)
print("Firms:",EXPECTED_FIRMS,"| Sectors:",EXPECTED_SECTORS)
print("Neural seeds:",NEURAL_SEEDS)
print("MCS/SPA reps:",MCS_SPA_REPS,"| CI bootstrap reps:",BOOTSTRAP_REPS,
      "| wild-cluster reps:",WILD_CLUSTER_REPS)
print("IMPORTANT: run separate R WINDOW, DISTRIBUTION and X profiles for the econometric sensitivity appendix.")

# Convenience package of the completed Python-stage outputs. Checkpoints are
# deliberately excluded because they are recovery artefacts, not manuscript
# results.
# Finalise reproducibility metadata before zipping.
with open(ENV_DIR / "RUN_COMPLETED_UTC.txt", "w", encoding="utf-8") as f:
    f.write(pd.Timestamp.utcnow().isoformat())

# Save exact code/runtime metadata in the bundle.
repro_meta = pd.DataFrame([
    {"Item":"Code_Version","Value":CODE_VERSION},
    {"Item":"Single_Session_From_Scratch","Value":True},
    {"Item":"Google_Drive_Used","Value":False},
    {"Item":"Prior_Model_Outputs_Reused","Value":False},
    {"Item":"Exact_Inputs_Bundled","Value":INCLUDE_EXACT_INPUTS_IN_REPRO_BUNDLE},
    {"Item":"R_Input_SHA256","Value":sha256_file(R_INPUT)},
    {"Item":"Master_Input_SHA256","Value":sha256_file(MASTER_FILE)},
    {"Item":"Python","Value":sys.version},
    {"Item":"TensorFlow","Value":tf.__version__},
    {"Item":"Platform","Value":platform.platform()},
])
save_csv(repro_meta,"34_SingleSession_Reproducibility_Meta.csv")

# Final manifest is generated after all reproducibility metadata are written.
final_manifest=[]
for pth in sorted(OUTPUT_DIR.rglob("*")):
    if not pth.is_file() or pth.name.endswith(".tmp"):
        continue
    rel=pth.relative_to(OUTPUT_DIR)
    if "_checkpoints_v5_09" in rel.parts:
        continue
    final_manifest.append({
        "Relative_Path":str(rel),
        "Bytes":pth.stat().st_size,
        "SHA256":sha256_file(pth),
    })
save_csv(pd.DataFrame(final_manifest),"35_Final_Reproducibility_Manifest.csv")

# Build one clean bundle. Local checkpoint files are transient recovery
# artefacts and are excluded; all manuscript inputs/results/configuration,
# figures, hashes and environment metadata are included.
REPRO_ZIP = Path("NSE_Reviewer_Aligned_Python_v5_09_REPRODUCIBILITY_BUNDLE.zip")
with zipfile.ZipFile(REPRO_ZIP,"w",compression=zipfile.ZIP_DEFLATED) as zf:
    for pth in sorted(OUTPUT_DIR.rglob("*")):
        if not pth.is_file() or pth.name.endswith(".tmp"):
            continue
        rel = pth.relative_to(OUTPUT_DIR)
        if "_checkpoints_v5_09" in rel.parts:
            continue
        zf.write(pth,arcname=str(rel))

print("Reproducibility bundle:", REPRO_ZIP.resolve())

if _running_in_colab() and AUTO_DOWNLOAD_REPRO_BUNDLE_IN_COLAB:
    try:
        from google.colab import files
        files.download(str(REPRO_ZIP))
        print("Automatic browser download started.")
    except Exception as ex:
        print("Automatic download could not start:", ex)
        print("Use the Colab Files panel to download:", REPRO_ZIP)




A Google Colab upload window will open now.
Please select BOTH current analysis files:
  1. TGARCH_X_Reviewer_Aligned_20160801_20260731_PRIMARY.zip
  2. Final Master File_All variables 01082016-31072026.csv



Saving Final Master File_All variables 01082016-31072026.csv to Final Master File_All variables 01082016-31072026.csv
Saving TGARCH_X_Reviewer_Aligned_20160801_20260731_PRIMARY.zip to TGARCH_X_Reviewer_Aligned_20160801_20260731_PRIMARY.zip

Uploaded files for this run:
   Final Master File_All variables 01082016-31072026.csv
   TGARCH_X_Reviewer_Aligned_20160801_20260731_PRIMARY.zip
Using freshly uploaded R ZIP: TGARCH_X_Reviewer_Aligned_20160801_20260731_PRIMARY.zip
Using freshly uploaded master CSV: Final Master File_All variables 01082016-31072026.csv
Exact inputs copied into reproducibility bundle:
   NSE_Reviewer_Aligned_Python_Output_v5_09_SingleSession/inputs/TGARCH_X_Reviewer_Aligned_20160801_20260731_PRIMARY.zip
   NSE_Reviewer_Aligned_Python_Output_v5_09_SingleSession/inputs/Final Master File_All variables 01082016-31072026.csv
Single-session mode: Google Drive persistence is DISABLED.
Single-session mode: prior v5.05-v5.08 outputs will NOT be reused.
Local output directory: 

PatchTST tune+fit: 100%|██████████| 11/11 [24:24<00:00, 133.14s/it]



PRIMARY REGIME MODEL: ZERO-NEUTRAL STUDENT-t HMM WITH SOFT PROBABILITIES


Primary HMM final-fit: 100%|██████████| 11/11 [00:05<00:00,  2.01it/s]


...

Transformer tune+fit: 100%|██████████| 11/11 [48:15<00:00, 263.21s/it]



PIPELINE COMPLETE
Output: /content/NSE_Reviewer_Aligned_Python_Output_v5_09_SingleSession
R version verified: tgarch_x_reviewer_aligned_v4_04_2016_2026
Python version: nse_reviewer_aligned_python_v5_09_single_session_reproducible_2016_2026
Firms: 52 | Sectors: 11
Neural seeds: [11, 23, 42, 71, 101]
MCS/SPA reps: 5000 | CI bootstrap reps: 5000 | wild-cluster reps: 999
IMPORTANT: run separate R WINDOW, DISTRIBUTION and X profiles for the econometric sensitivity appendix.
Reproducibility bundle: /content/NSE_Reviewer_Aligned_Python_v5_09_REPRODUCIBILITY_BUNDLE.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Automatic browser download started.
